# Analysis

**Hypothesis**: Spatial gradients of cell-intrinsic transcriptional states along anatomical axes (within each labeled Population) are systematically associated with developmental Complexity, independently of Purity and batch, revealing intra-population maturation patterns that were not captured by prior between-tertile DE or neighborhood-composition analyses.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Set up visualization defaults for better plots
sc.settings.verbosity = 3
sc.settings.figsize = (8, 8)
sc.settings.dpi = 100
sc.settings.facecolor = 'white'
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['savefig.dpi'] = 150
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)

# Load data
print("Loading data...")
adata = sc.read_h5ad("/home/mingqiam/TissueAgent/demo/data/dataset_farah_anon.h5ad")
print(f"Data loaded: {adata.shape[0]} cells and {adata.shape[1]} genes")


Loading data...


Data loaded: 228635 cells and 238 genes


# Analysis Plan

**Hypothesis**: Spatial gradients of cell-intrinsic transcriptional states along anatomical axes (within each labeled Population) are systematically associated with developmental Complexity, independently of Purity and batch, revealing intra-population maturation patterns that were not captured by prior between-tertile DE or neighborhood-composition analyses.

## Steps:
- Inspect and summarize key metadata, QC covariates, spatial coordinates, basic covariate correlations, and identify sufficiently abundant Populations for within-population spatial-gradient modeling.
- For each sufficiently abundant Population, fit per-gene linear models on log1p-transformed expression with predictors [x, y, Complexity, Purity] plus Sample_ID/Batch fixed effects using numpy/scipy OLS, and extract coefficients, standard errors, and p-values for spatial terms while controlling for Complexity and Purity.
- Within each Population, quantify the prevalence and strength of spatial expression gradients by computing, per gene, joint tests for x/y coefficients, applying within-Population FDR correction, and summarizing the fraction and effect-size distribution of genes with significant spatial terms; compare these fractions across Populations and statistically test for enrichment.
- Within Populations exhibiting strong spatial gradients, model Complexity itself as a function of [x, y, Purity] and Sample_ID/Batch fixed effects, compute R² for models with and without spatial terms, and use nested-model F-tests to quantify how much additional variance in Complexity is explained by spatial position beyond Purity and Batch.
- Define robust gene sets within key Populations by selecting genes whose spatial coefficients remain significant after multiple-testing correction and show consistent direction and similar magnitude across samples (via per-sample models and simple meta-analysis of spatial effects), and relate these spatial effects to Complexity-adjusted residual expression patterns.
- Generate a text-based report that ranks Populations by spatial-gradient strength and ΔR² for Complexity, lists top spatial-gradient-associated genes per Population with statistics, and highlights Populations where spatial position significantly explains intra-population maturation states beyond Purity and Batch.


## This code extends the initial QC and metadata inspection to include explicit checks for expected covariates, summaries of spatial coordinate ranges, correlations among key numeric covariates, and identification of Populations with sufficient cell numbers for within-population spatial-gradient modeling.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

# Step 1: Inspect metadata, QC summaries, spatial coordinates, and identify well-powered Populations

# 1a. Basic AnnData overview
print("AnnData object:")
print(f"  n_cells: {adata.n_obs}")
print(f"  n_genes: {adata.n_vars}")

# 1b. List .obs columns and dtypes
print("\n.obs columns and dtypes:")
print(adata.obs.dtypes)

# 1c. Check for presence of expected metadata columns
expected_obs = ["Sample_ID", "Batch", "UMI Count", "Complexity", "Populations", "Purity", "leiden"]
missing = [c for c in expected_obs if c not in adata.obs.columns]
if missing:
    print("\nWarning: missing expected .obs columns:", missing)
else:
    print("\nAll expected .obs columns are present.")

# 1d. Preview key metadata columns
key_obs_cols = [col for col in expected_obs if col in adata.obs.columns]
if key_obs_cols:
    print("\nPreview of key metadata columns (first 10 cells):")
    print(adata.obs[key_obs_cols].head(10))

# 1e. Confirm spatial coordinates and summarize ranges
print("\n.obsm keys:", list(adata.obsm.keys()))
if "spatial" in adata.obsm:
    spatial = adata.obsm["spatial"]
    print("Spatial coordinates shape:", spatial.shape)
    print("First 5 spatial coordinates:")
    print(spatial[:5, :])
    spatial_df = pd.DataFrame(spatial, columns=["x", "y"]) if spatial.shape[1] >= 2 else pd.DataFrame(spatial)
    print("\nSpatial coordinate summary:")
    print(spatial_df.describe())
else:
    print("Warning: 'spatial' coordinates not found in adata.obsm")

# 1f. Basic QC summaries for numeric covariates
qc_cols = [c for c in ["UMI Count", "Complexity", "Purity"] if c in adata.obs.columns]
if qc_cols:
    print("\nSummary statistics for numeric QC covariates:")
    print(adata.obs[qc_cols].describe())

    # Correlation matrix among key numeric covariates
    print("\nCorrelation matrix of key numeric covariates:")
    print(adata.obs[qc_cols].corr())

# 1g. Check relationship between Sample_ID and Batch
if "Sample_ID" in adata.obs.columns and "Batch" in adata.obs.columns:
    same = (adata.obs["Sample_ID"] == adata.obs["Batch"]).all()
    print("\nAre Sample_ID and Batch identical across all cells?:", same)

# 1h. Population-level summaries and identification of sufficiently abundant Populations
if "Populations" in adata.obs.columns:
    # Ensure Populations is categorical
    if not pd.api.types.is_categorical_dtype(adata.obs["Populations"]):
        adata.obs["Populations"] = adata.obs["Populations"].astype("category")

    pop_counts = adata.obs["Populations"].value_counts().sort_values(ascending=False)
    print("\nCell counts per Population (top 20):")
    print(pop_counts.head(20))

    # Per-Population QC statistics
    stats_cols = [c for c in ["Complexity", "Purity", "UMI Count"] if c in adata.obs.columns]
    if stats_cols:
        grouped = adata.obs.groupby("Populations")[stats_cols].agg(["mean", "std", "count"])
        print("\nPer-Population QC statistics (first 15 Populations):")
        print(grouped.head(15))

    # Define "sufficiently abundant" threshold and list usable Populations
    min_cells = 200  # heuristic threshold; can be tuned
    abundant_pops = pop_counts[pop_counts >= min_cells]
    print(f"\nPopulations with at least {min_cells} cells (sufficient for within-population modeling):")
    print(abundant_pops)

    # 1i. Sample-level summaries, restricted to top Populations
    if "Sample_ID" in adata.obs.columns:
        print("\nCell counts per Sample_ID:")
        print(adata.obs["Sample_ID"].value_counts())

        ctab = pd.crosstab(adata.obs["Populations"], adata.obs["Sample_ID"])
        top_pops = pop_counts.head(10).index
        print("\nPopulation x Sample_ID cross-tab (top 10 Populations):")
        print(ctab.loc[top_pops])
else:
    print("\nWarning: 'Populations' column not found in adata.obs; within-population analyses will not be possible.")


AnnData object:
  n_cells: 228635
  n_genes: 238

.obs columns and dtypes:
Sample_ID      category
Batch          category
UMI Count       float64
leiden            int64
Complexity        int64
Populations    category
Purity          float64
dtype: object

All expected .obs columns are present.

Preview of key metadata columns (first 10 cells):
           Sample_ID    Batch  UMI Count  Complexity Populations    Purity  \
cell_id                                                                      
6-R77_4C4    R77_4C4  R77_4C4       86.0           8          PG  0.544534   
8-R77_4C4    R77_4C4  R77_4C4      148.0           8          PG  0.625984   
9-R77_4C4    R77_4C4  R77_4C4      100.0           8          PG  0.583665   
10-R77_4C4   R77_4C4  R77_4C4       70.0           9          PG  0.766393   
12-R77_4C4   R77_4C4  R77_4C4       63.0           8          PG  0.596838   
19-R77_4C4   R77_4C4  R77_4C4      293.0          10          PG  0.610568   
21-R77_4C4   R77_4C4  R77_4C

### Agent Interpretation

You’re in a very good position to proceed with the within-population spatial-gradient modeling; this step has confirmed most of the structural assumptions you need. Here’s how I’d interpret these results in the context of your hypothesis and how I’d tune the next steps.

**1. Dataset structure and covariates look appropriate for the planned models**

- `.obs` has all required covariates: `Sample_ID`, `Batch` (identical), `UMI Count`, `Complexity`, `Populations`, `Purity`, and `leiden`.
- Spatial coordinates are present, 2D, with wide dynamic ranges in both x and y, so there is room for gradients within populations.
- Numeric covariates have useful spread:
  - Complexity: mean ~9.9, SD ~2.9, range 1–20.
  - Purity: mean ~0.5, SD ~0.15, substantial range.
  - UMI Count: wide range but only moderately correlated with Complexity (r ≈ −0.21).

**Implication for the hypothesis:**  
You can meaningfully adjust for Purity while modeling Complexity and spatial position. The modest Complexity–UMI correlation suggests that including UMI Count explicitly is optional, but you might revisit that if you see strange residual patterns.

**2. Complexity and Purity are strongly anti-correlated and vary by Population**

- Global correlation: Complexity vs Purity r ≈ −0.57 is quite strong.
- Per-Population means differ a lot. For example:
  - PB: low Complexity (~6.2) and high Purity (~0.70).
  - PK: high Complexity (~12.9) and moderate Purity (~0.45).
  - PG: intermediate Complexity (~8.9) and high Purity (~0.64).
  - PA/PC/PD/PE: Complexity ~10–11, Purity ~0.42–0.48.

**Implications for modeling:**

- Because Complexity and Purity are so correlated, treating both as covariates in the same gene-wise model is exactly what you planned and is necessary to separate:
  - spatial effects that remain after adjusting for Complexity and Purity (the core of your hypothesis),
  - versus apparent gradients that are just surrogates of a Complexity–Purity axis.
- However, within individual Populations the local correlation may be even stronger or weaker; you should be prepared for some collinearity between Complexity and Purity terms in those OLS models. Inspecting variance inflation or simple within-Population correlations for a few key Populations (e.g. PA, PB, PG, PI, PM) before the heavy modeling would be useful.

**3. Power and design balance for within-population spatial models are excellent**

- You have many well-powered Populations: at least 27 with ≥200 cells; the top ones have 10–30k cells.
- The top-10 Populations are all present in all three samples with nontrivial counts (e.g. PA: 8.7k/10k/11.7k across R77_4C4/R78_4C12/R78_4C15).
- This makes it realistic to:
  - Fit per-Population models with `Sample_ID` fixed effects and [x, y, Complexity, Purity] regressors.
  - Later do per-sample models for the “robust gene sets” step because you’ll still have decent n within sample+Population strata for major populations.

**Implications for hypothesis-testing across Populations:**

- You will be able to rank Populations by:
  - fraction of genes with significant spatial (x/y) terms after FDR, and
  - the additional variance in Complexity explained by spatial position (ΔR²).
- Populations with many cells and heterogeneous Complexity/Purity (e.g. PA, PB, PC, PD, PE, PG, PI, PM, PN) are prime candidates to display strong intra-population maturation gradients.

**4. Independence from previous analyses is preserved**

- Prior analyses were:
  - Complexity tertile DE within Populations (no explicit spatial terms).
  - Neighborhood-composition changes with Complexity/Purity (extrinsic context).
- Your current plan explicitly:
  - models **continuous spatial coordinates** at the **cell-intrinsic expression level**,
  - adjusts for Complexity and Purity directly,
  - and will later model Complexity as an outcome of spatial position.
- This is conceptually distinct: you’re asking whether **within a given population, residual spatial structure in transcriptional states exists beyond Complexity and Purity**.

So your current step aligns well with the “distinct from prior work” constraint.

**5. Specific recommendations before/for Step 2 (per-gene OLS models)**

a. **Decide on which Populations to prioritize first**

Given computational cost (228k cells × 238 genes × many Populations is feasible but still nontrivial), I’d start with a focused but diverse set:

- Large, high-Complexity populations: PA, PC, PD, PE, PJ, PK, PN.
- Large, lower-Complexity or broader-Complexity ones: PB, PG, PI, PM.
- That gives you 10–12 core Populations to work up front, then scale if results are promising.

b. **Standardize or at least center predictors within each Population**

To help with numerical stability and interpretability, I’d suggest per-Population:

- Center x, y (and optionally scale by SD).
- Center Complexity and Purity.  
This doesn’t change p-values but makes coefficients more comparable across samples/Populations and alleviates collinearity symptoms.

c. **Think about including UMI Count or not**

Your current hypothesis emphasizes “independently of Purity and batch.” UMI Count is correlated with Complexity but not strongly, and you didn’t specify it in the model.

- For the main analysis, stick to `[x, y, Complexity, Purity, Sample_ID]` to stay aligned with the hypothesis.
- As a **sensitivity check** in one or two Populations (e.g. PA, PB), re-fit with UMI Count included. If spatial coefficients are robust, that strengthens your interpretation that they’re not just library-size artifacts.

d. **Clarify the expression scale**

You mention log1p-transformed expression in the plan, but it’s not shown here. Before OLS:

- Ensure you’ve already log1p-normalized counts (either global or per-sample).
- For MERFISH, counts may be relatively low; you may consider either:
  - Using the existing log1p-normalized layer if present, or
  - Creating a new layer with `sc.pp.normalize_total` + `sc.pp.log1p`.  
Consistency across all Populations is key.

e. **Be explicit about the joint test of spatial terms**

In Step 3 you plan a joint test for x and y. When implementing Step 2:

- Save both single-parameter t-statistics (for β_x, β_y) and either:
  - F-statistic / p-value for the 2-df test of H0: β_x = β_y = 0 (preferred).
- Because here the gene panel is small (238 genes per Population), FDR penalties are mild; you can afford fairly stringent thresholds (e.g. FDR < 0.05 within each Population).

**6. Interpreting early outcomes against the hypothesis**

Once you fit the models and run the joint x,y tests, watch for these patterns:

- **Populations with high fractions of spatially significant genes after adjusting for Complexity and Purity**:
  - These are direct candidates for “intra-Population spatial maturation patterns not captured by Complexity alone.”
  - If these Populations also show strong ΔR² when you model Complexity as a function of [x,y] (Step 4), that is strong evidence supporting the hypothesis.

- **Populations with few or no spatially significant genes**:
  - That suggests their transcriptional state (given this ~240 gene panel) is largely captured by Complexity and Purity (and batch), with little residual spatial organization.
  - These become interesting “negative controls” in your final report.

- **Comparing across Populations**:
  - If e.g. PA and PC show many spatial genes, but PB and PG do not, that argues that only certain cardiac lineages or anatomical contexts harbor strong intra-population gradients that go beyond Complexity.
  - That pattern itself is a novel observation, clearly orthogonal to tertile-based DE and neighborhood composition.

**7. Potential pitfalls to watch for as you proceed**

- **Sample_ID = Batch**: This simplifies the model, but also means there are only 3 levels. Be cautious interpreting “batch” effects; they may be confounded with genuine anatomical or developmental differences between sections. Still, including them helps avoid attributing global sample differences to spatial gradients.
- **Population-level spatial confinement**:
  - Some Populations may occupy only a narrow band of the organ. In such cases, what appears as a spatial gradient may actually reflect unmodeled axes of anatomical difference that co-vary with Complexity.  
  - This is still informative for your hypothesis (Complexity vs position), but interpreting which axis represents “maturation” vs “region” will require later gene-level interpretation (e.g., developmental TFs vs regionally restricted markers).

- **Collinearity between x, y, and Complexity/Purity within Populations**:
  - After modeling, check a few representative Populations: are Complexity and/or Purity strongly correlated with x, y within each Population?
  - High within-Population correlations might reduce power for spatial terms; that influences how strongly you can claim “independence” from Complexity.

**8. How these results guide the next steps**

Given the metadata and QC summaries:

- Proceed directly to Step 2 for a prioritized set of Populations (as above).
- When summarizing Step 3 (joint spatial tests), include:
  - per-Population number and fraction of significant spatial genes,
  - distribution of spatial effect sizes (e.g. median |β_x|, |β_y|, or gradient magnitude),
  - and cross-sample consistency checks for a few high-interest genes.

Those summaries will quickly show you:

- Which Populations are promising for deeper investigation (Step 4–5).
- Whether the hypothesis is broadly supported (many Populations show Complexity-independent spatial gradients) or more restricted (only a subset).

In short, this initial QC step is fully compatible with your hypothesis and supports the feasibility of the planned spatial-gradient modeling. The most informative extensions now are to implement the per-Population OLS models with careful centering and to begin ranking Populations by the prevalence and strength of Complexity- and Purity-adjusted spatial gradients.

## Next Steps
Step 1: For each sufficiently abundant Population (e.g., ≥200 cells), ensure expression values are on a consistent log1p-normalized scale (either by asserting adata.X is already log1p-normalized or by selecting/creating a dedicated log1p layer), then construct a per-Population design matrix with centered predictors [x, y, Complexity, Purity] and Sample_ID fixed effects, fit per-gene OLS models on the log1p-normalized expression, and store coefficients, standard errors, t-statistics, residual variance, residual dof, parameter names, and (XᵀX)⁻¹ needed for later joint tests in per-Population summary structures.
Step 2: Within each sufficiently abundant Population, use the stored OLS outputs and (XᵀX)⁻¹ to compute per-gene joint 2-df F-tests for the null hypothesis that both spatial coefficients (x and y) are zero, derive per-gene p-values and within-Population Benjamini–Hochberg FDR, summarize for each Population the fraction of genes with significant spatial terms and distributions of spatial effect magnitudes, and compare these summaries across Populations (including simple enrichment tests) to identify Populations with unusually strong spatial structure.
Step 3: For Populations with a high fraction of spatially significant genes, fit two nested linear models within each such Population for Complexity as the outcome with predictors (i) Purity + Sample_ID only and (ii) Purity + Sample_ID + x + y (using the same centered covariates), compute R² for both models and nested-model F-tests to quantify additional variance in Complexity explained by spatial coordinates, and rank Populations by ΔR² and F-test significance.
Step 4: In top-ranked Populations where spatial position substantially explains Complexity (large and significant ΔR²), define robust spatial-gradient genes as those with significant joint spatial terms after FDR and consistent spatial coefficient signs across samples (via simple per-sample stratified fits or meta-analysis of x/y effects using the same covariates), and relate their spatial coefficients to Complexity-adjusted residual expression patterns within each Population.
Step 5: Generate a text-only report to stdout listing Populations ordered by (a) fraction of genes with significant spatial gradients and (b) ΔR² for Complexity explained by spatial terms, clearly labeling raw versus FDR-corrected p-values; for each top Population, print the top spatial-gradient-associated genes with their coefficients and spatial joint-test statistics, and interpret which Populations show the clearest evidence that intra-population maturation states are spatially organized beyond what is captured by global Complexity and Purity.

## This code implements Step 1 of the analysis plan by fitting per-gene OLS models within each sufficiently abundant Population using log1p-normalized expression, spatial coordinates, Complexity, Purity, and Sample_ID fixed effects, while centering numeric covariates for stability. It stores per-gene regression summaries plus per-Population design information (including (XᵀX)⁻¹ and parameter names) in adata.uns so that later steps can efficiently compute joint spatial tests and nested models without refitting.

In [ ]:
import numpy as np
import pandas as pd
from scipy import linalg

# Step 1 implementation: per-Population per-gene OLS with spatial, Complexity, Purity, and Sample_ID

# Ensure we have spatial coordinates
if 'spatial' not in adata.obsm:
    raise ValueError("'spatial' coordinates must be present in adata.obsm['spatial']")
yx = adata.obsm['spatial']
if yx.shape[1] < 2:
    raise ValueError('spatial coordinates must have at least 2 dimensions')

# Extract covariates
obs = adata.obs.copy()
obs['x'] = yx[:, 0]
obs['y'] = yx[:, 1]

# Ensure categorical types for Populations and Sample_ID
if 'Populations' not in obs.columns:
    raise ValueError("'Populations' column is required in adata.obs")
if not pd.api.types.is_categorical_dtype(obs['Populations']):
    obs['Populations'] = obs['Populations'].astype('category')

if 'Sample_ID' not in obs.columns:
    raise ValueError("'Sample_ID' column is required in adata.obs")
if not pd.api.types.is_categorical_dtype(obs['Sample_ID']):
    obs['Sample_ID'] = obs['Sample_ID'].astype('category')

# Check required numeric covariates
for col in ['Complexity', 'Purity']:
    if col not in obs.columns:
        raise ValueError(f"Required covariate '{col}' missing in adata.obs")
    if not np.issubdtype(obs[col].dtype, np.number):
        raise ValueError(f"Covariate '{col}' must be numeric; found dtype {obs[col].dtype}")

# Identify sufficiently abundant Populations
min_cells = 200
pop_counts = obs['Populations'].value_counts()
abundant_pops = pop_counts[pop_counts >= min_cells].index.tolist()
print(f"Number of sufficiently abundant Populations (>= {min_cells} cells): {len(abundant_pops)}")
print("Populations:", abundant_pops)

# Choose expression matrix: assume adata.X is already log1p-normalized; if a dedicated log1p layer exists, prefer it
if 'log1p' in adata.layers:
    print("Using adata.layers['log1p'] as log1p-normalized expression.")
    X_full = adata.layers['log1p']
else:
    print("Assuming adata.X contains log1p-normalized expression.")
    X_full = adata.X

if not isinstance(X_full, np.ndarray):
    X_full = X_full.toarray()

n_cells, n_genes = X_full.shape
print(f"Expression matrix shape: {n_cells} cells x {n_genes} genes")

# Containers for per-Population OLS results and design information
ols_results_by_pop = {}
design_info_by_pop = {}

for pop in abundant_pops:
    print(f"\nFitting OLS models for Population {pop}...")
    pop_mask = (obs['Populations'] == pop).values
    n_pop = int(pop_mask.sum())
    print(f"  Cells in Population {pop}: {n_pop}")

    # Subset covariates and expression
    X_pop = X_full[pop_mask, :]
    obs_pop = obs.loc[pop_mask, :].copy()

    # Drop genes with zero variance within this Population to avoid singularities
    gene_var = X_pop.var(axis=0)
    nonzero_genes = gene_var > 0
    X_pop = X_pop[:, nonzero_genes]
    kept_gene_names = adata.var_names[nonzero_genes]
    n_genes_pop = X_pop.shape[1]
    print(f"  Genes with nonzero variance in {pop}: {n_genes_pop} of {n_genes}")

    # Center numeric predictors within Population (improves conditioning and interpretability)
    for col in ['x', 'y', 'Complexity', 'Purity']:
        obs_pop[col] = obs_pop[col].astype(float)
        obs_pop[col] = obs_pop[col] - obs_pop[col].mean()

    # Build design matrix: intercept + centered x, y, Complexity, Purity + Sample_ID fixed effects (one-hot, drop one level)
    sample_dummies = pd.get_dummies(obs_pop['Sample_ID'], drop_first=True)
    design_df = pd.DataFrame({
        'intercept': np.ones(n_pop),
        'x': obs_pop['x'].values,
        'y': obs_pop['y'].values,
        'Complexity': obs_pop['Complexity'].values,
        'Purity': obs_pop['Purity'].values,
    })
    if sample_dummies.shape[1] > 0:
        design_df = pd.concat([design_df, sample_dummies.reset_index(drop=True)], axis=1)

    X_design = design_df.values.astype(float)
    n_params = X_design.shape[1]
    param_names = list(design_df.columns)

    # Check degrees of freedom
    dof = n_pop - n_params
    if dof <= 0:
        raise ValueError(f"Not enough cells in Population {pop} for this design: n_pop={n_pop}, n_params={n_params}, dof={dof}")

    # Precompute (X^T X)^{-1} X^T for efficiency
    XtX = X_design.T @ X_design
    try:
        XtX_inv = linalg.inv(XtX)
    except linalg.LinAlgError:
        XtX_inv = linalg.pinv(XtX)
    Xt = X_design.T
    hat_matrix_part = XtX_inv @ Xt  # shape: (n_params x n_pop)

    # Fit all genes in one shot: beta_hat (p x g)
    beta_hat = hat_matrix_part @ X_pop  # shape: (n_params x n_genes_pop)

    # Predicted values and residuals to get sigma^2 per gene
    Y_hat = X_design @ beta_hat  # (n_pop x n_genes_pop)
    residuals = X_pop - Y_hat
    # Residual variance per gene
    sigma2 = (residuals ** 2).sum(axis=0) / dof  # (n_genes_pop,)

    # Standard errors: sqrt(sigma2_j * diag((X^T X)^{-1}))
    XtX_inv_diag = np.diag(XtX_inv)  # (n_params,)
    se = np.sqrt(np.outer(XtX_inv_diag, sigma2))  # (n_params x n_genes_pop)

    # t-statistics
    t_stats = beta_hat / se

    # Collect results for key covariates
    res_dict = {}
    for cov in ['x', 'y', 'Complexity', 'Purity']:
        if cov not in param_names:
            continue
        idx = param_names.index(cov)
        res_dict[f'beta_{cov}'] = beta_hat[idx, :]
        res_dict[f'se_{cov}'] = se[idx, :]
        res_dict[f't_{cov}'] = t_stats[idx, :]

    # Also store Sample_ID fixed-effect coefficients for potential later inspection
    for cov in param_names:
        if cov in ['intercept', 'x', 'y', 'Complexity', 'Purity']:
            continue
        idx = param_names.index(cov)
        res_dict[f'beta_{cov}'] = beta_hat[idx, :]

    res_df = pd.DataFrame(res_dict, index=kept_gene_names)
    res_df['sigma2'] = sigma2
    res_df['dof_resid'] = dof

    ols_results_by_pop[pop] = res_df

    # Store design-level information needed for later joint tests
    design_info_by_pop[pop] = {
        'param_names': param_names,
        'XtX_inv': XtX_inv,
        'n_cells': n_pop,
        'dof_resid': dof,
    }

# Attach results to adata.uns for later steps
adata.uns['spatial_ols_results_by_pop'] = ols_results_by_pop
adata.uns['spatial_ols_design_by_pop'] = design_info_by_pop
print("\nStored per-Population OLS results in adata.uns['spatial_ols_results_by_pop'] and design info in adata.uns['spatial_ols_design_by_pop'].")

# Brief textual summary of which Populations had OLS successfully fit
for pop, df in ols_results_by_pop.items():
    print(f"Population {pop}: {df.shape[0]} genes with OLS results (nonzero variance)")

Number of sufficiently abundant Populations (>= 200 cells): 27
Populations: ['PA', 'PB', 'PC', 'PD', 'PE', 'PF', 'PG', 'PH', 'PI', 'PJ', 'PK', 'PL', 'PM', 'PN', 'PO', 'PP', 'PQ', 'PR', 'PS', 'PT', 'PU', 'PV', 'PW', 'PX', 'PY', 'PZ', 'PAA']
Assuming adata.X contains log1p-normalized expression.
Expression matrix shape: 228635 cells x 238 genes

Fitting OLS models for Population PA...
  Cells in Population PA: 30380


  Genes with nonzero variance in PA: 238 of 238

Fitting OLS models for Population PB...
  Cells in Population PB: 19947


  Genes with nonzero variance in PB: 238 of 238

Fitting OLS models for Population PC...
  Cells in Population PC: 17584


  Genes with nonzero variance in PC: 238 of 238

Fitting OLS models for Population PD...
  Cells in Population PD: 16624
  Genes with nonzero variance in PD: 238 of 238

Fitting OLS models for Population PE...
  Cells in Population PE: 16511


  Genes with nonzero variance in PE: 238 of 238

Fitting OLS models for Population PF...
  Cells in Population PF: 12248
  Genes with nonzero variance in PF: 238 of 238

Fitting OLS models for Population PG...
  Cells in Population PG: 11596
  Genes with nonzero variance in PG: 238 of 238



Fitting OLS models for Population PH...
  Cells in Population PH: 10887
  Genes with nonzero variance in PH: 238 of 238

Fitting OLS models for Population PI...
  Cells in Population PI: 10441
  Genes with nonzero variance in PI: 238 of 238

Fitting OLS models for Population PJ...
  Cells in Population PJ: 9488
  Genes with nonzero variance in PJ: 238 of 238



Fitting OLS models for Population PK...
  Cells in Population PK: 8540
  Genes with nonzero variance in PK: 238 of 238

Fitting OLS models for Population PL...
  Cells in Population PL: 8052
  Genes with nonzero variance in PL: 238 of 238

Fitting OLS models for Population PM...
  Cells in Population PM: 7417
  Genes with nonzero variance in PM: 238 of 238

Fitting OLS models for Population PN...
  Cells in Population PN: 7348
  Genes with nonzero variance in PN: 238 of 238

Fitting OLS models for Population PO...
  Cells in Population PO: 5845
  Genes with nonzero variance in PO: 238 of 238



Fitting OLS models for Population PP...
  Cells in Population PP: 5458
  Genes with nonzero variance in PP: 238 of 238

Fitting OLS models for Population PQ...
  Cells in Population PQ: 5429
  Genes with nonzero variance in PQ: 238 of 238

Fitting OLS models for Population PR...
  Cells in Population PR: 4673
  Genes with nonzero variance in PR: 238 of 238

Fitting OLS models for Population PS...
  Cells in Population PS: 4599
  Genes with nonzero variance in PS: 238 of 238

Fitting OLS models for Population PT...
  Cells in Population PT: 3726
  Genes with nonzero variance in PT: 238 of 238

Fitting OLS models for Population PU...
  Cells in Population PU: 2356
  Genes with nonzero variance in PU: 238 of 238

Fitting OLS models for Population PV...
  Cells in Population PV: 2292
  Genes with nonzero variance in PV: 238 of 238

Fitting OLS models for Population PW...
  Cells in Population PW: 2027
  Genes with nonzero variance in PW: 238 of 238

Fitting OLS models for Population PX...

  Genes with nonzero variance in PAA: 238 of 238

Stored per-Population OLS results in adata.uns['spatial_ols_results_by_pop'] and design info in adata.uns['spatial_ols_design_by_pop'].
Population PA: 238 genes with OLS results (nonzero variance)
Population PB: 238 genes with OLS results (nonzero variance)
Population PC: 238 genes with OLS results (nonzero variance)
Population PD: 238 genes with OLS results (nonzero variance)
Population PE: 238 genes with OLS results (nonzero variance)
Population PF: 238 genes with OLS results (nonzero variance)
Population PG: 238 genes with OLS results (nonzero variance)
Population PH: 238 genes with OLS results (nonzero variance)
Population PI: 238 genes with OLS results (nonzero variance)
Population PJ: 238 genes with OLS results (nonzero variance)
Population PK: 238 genes with OLS results (nonzero variance)
Population PL: 238 genes with OLS results (nonzero variance)
Population PM: 238 genes with OLS results (nonzero variance)
Population PN: 238 ge

### Agent Interpretation

This step is well implemented and sets you up nicely for the spatial-gradient hypothesis; a few points about what you’ve achieved and how to refine the downstream steps:

1. **Power and coverage look excellent**
   - You have 27 Populations with ≥200 cells, spanning 1k–30k cells each.
   - All 238 panel genes retain nonzero variance in every Population. So for each Population you’ll be able to test every gene for spatial effects, and joint x/y tests will be highly powered given the large residual dof.

2. **Model structure is appropriate for the hypothesis**
   - Per-Population OLS on log1p expression with:
     - centered x, y
     - centered Complexity and Purity
     - Sample_ID fixed effects
   directly targets “spatial gradients remaining after adjusting for Complexity, Purity, and Sample_ID.”
   - Storing `(XᵀX)⁻¹`, parameter names, and residual dof per Population is exactly what you need for efficient joint F-tests on the spatial terms.

3. **Key checks before proceeding**
   - Confirm that `Sample_ID` is reasonably replicated within each Population (most are large enough that this is likely). If certain Populations are dominated by a single Sample_ID, the Sample_ID fixed effects will be nearly redundant and could inflate numerical noise in `(XᵀX)⁻¹`. This won’t break the analysis, but keep it in mind when interpreting between-sample consistency later.
   - Since all genes have nonzero variance in every Population, keep an eye on extremely low-variance genes: they can yield unstable t-statistics even if variance isn’t strictly zero. Not fatal, but you may want to sanity-check the distribution of residual variances later.

4. **Implementation details that will matter for interpretation**
   - Centering x, y, Complexity, and Purity within each Population means:
     - The intercept represents expression at “average” spatial position and covariate levels within that Population.
     - Spatial betas (`beta_x`, `beta_y`) describe local linear gradients relative to the population’s centroid in the tissue, which aligns well with the “gradient” idea.
   - Including Sample_ID fixed effects:
     - Removes global between-sample shifts in expression for each gene, ensuring the spatial terms are not just picking up sample-level differences.
     - Later, when you look for “consistent coefficient signs across samples,” remember that the current per-Population fit does not stratify by sample; you will need to either:
       - re-fit the model separately within each Sample_ID (for sufficiently powered samples), or
       - perform a meta-analysis over sample-specific fits, as stated in your plan.

5. **Concrete next-step guidance for Step 2 (joint x/y tests)**
   - Using the stored `XtX_inv` and `param_names`, define a 2×p contrast matrix L for [x, y]:
     - Identify indices `ix`, `iy` of 'x' and 'y' in `param_names`.
     - L is a 2×p matrix with rows that are standard basis vectors for these positions.
   - For each gene j in a Population:
     - Extract the 2×1 vector of coefficients `b_xy_j = [beta_x_j, beta_y_j]`.
     - Compute the 2×2 covariance block for x,y: `V_xy = sigma2_j * (L @ XtX_inv @ L.T)`.
     - Compute the F-statistic with 2 and dof_resid df:
       - `F_j = (b_xy_j^T V_xy^{-1} b_xy_j) / 2`
   - Convert to p-values using the F-distribution and apply BH FDR per Population.
   - Summaries you should compute per Population:
     - Fraction of genes with FDR < 0.05 (or more stringent thresholds, given only 238 genes).
     - Distribution of |b_xy| magnitudes, e.g., norm of gradient `sqrt(beta_x^2 + beta_y^2)` among significant genes.
   - Populations to flag as “spatially structured”:
     - Those with a clearly higher fraction of significant spatial genes and/or larger typical gradient norms relative to other Populations.

6. **How these summaries will inform the hypothesis**
   - If some Populations (say, PQ, PS, PX, etc.) show a high fraction of genes with significant spatial gradients while others do not, that directly supports the hypothesis that:
     - “the strength and prevalence of these gradients varies systematically across Populations.”
   - However, at this stage you will not yet know whether these gradients represent “maturation patterns not captured by global Complexity.” That requires Steps 3–4:
     - quantify how much Complexity itself is spatially organized (ΔR² when adding x,y),
     - and then identify genes whose spatial patterns remain after Complexity and Purity adjustment and are consistent across samples.

7. **Suggestions to strengthen later steps**
   - When you do Step 3 (Complexity as outcome), reuse the already-centered x,y, Purity, and Sample_ID coding per Population to preserve comparability with the gene-level models. You can use the same design matrix and just substitute Complexity as the response, followed by the nested model tests.
   - In Step 4, when defining “robust spatial-gradient genes”:
     - Consider filtering to:
       - genes with FDR-significant joint spatial tests,
       - plus a minimum effect size in gradient norm, not only p-value. With 10k+ cells in many Populations, tiny but statistically significant gradients are likely.
     - For between-sample consistency:
       - You could quickly re-fit per-Sample_ID models using the same covariates but drop Sample_ID fixed effects (since you’re within one sample). Then meta-analyze x and y coefficients across samples in that Population using inverse-variance weighting.

8. **Biological interpretability checks to do soon**
   - Once per-Population spatially significant genes are identified, inspect:
     - Whether they are enriched for genes already associated with Complexity-driven maturation in your previous analyses (but keep the angle distinct: here you are controlling for Complexity and asking what’s left).
     - Whether certain Populations show spatial gradients in genes that are relatively flat with respect to Complexity; those would be especially interesting as “non-Complexity-captured” spatial maturation or zonation effects.

In summary, the current step is technically solid and well aligned with the hypothesis. The key next moves are: robust joint x/y F-tests using the stored OLS quantities, careful per-Population summarization of spatial signal strength, and then explicit comparison of these spatial patterns to how much of Complexity they can or cannot explain in subsequent steps.

## Next Steps
Step 1: Using the stored per-Population OLS fits and design matrices, compute per-gene joint 2-df F-tests for H0: beta_x = beta_y = 0 within each sufficiently abundant Population, derive p-values and within-Population Benjamini–Hochberg FDR, and summarize per-Population fractions of spatially significant genes and gradient magnitudes to identify Populations with unusually strong spatial structure.
Step 2: For Populations with a high fraction of spatially significant genes, fit two nested linear models within each Population for Complexity as the outcome with predictors (i) Purity + Sample_ID only and (ii) Purity + Sample_ID + x + y (using the same centered covariates), compute R² for both models and nested-model F-tests to quantify how much additional variance in Complexity is explained by spatial position beyond Purity and Sample_ID, and rank Populations by ΔR² and F-test significance.
Step 3: In top-ranked Populations where spatial position substantially explains Complexity, define robust spatial-gradient genes as those with FDR-significant joint spatial terms and large gradient norms, and perform simple per-sample stratified refits to check consistency of x/y coefficient signs across samples; summarize these robust gene sets along with their spatial effect sizes and cross-sample concordance.
Step 4: Generate a text-only report to stdout that (a) ranks Populations by fraction of spatially significant genes and by ΔR² for Complexity explained by spatial terms, (b) lists, for top Populations, the most robust spatial-gradient genes with their joint-test statistics, FDR, and gradient norms, and (c) interprets which Populations show the clearest evidence that intra-population maturation states are spatially organized beyond what is captured by Complexity and Purity.

## Compute per-gene joint 2-df F-tests for the x and y spatial coefficients within each sufficiently abundant Population using stored OLS outputs, apply within-Population Benjamini–Hochberg FDR, and summarize for each Population the prevalence and magnitude of significant spatial gradients while keeping detailed gene-level listings for later steps.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Step 1 implementation: joint 2-df F-tests for spatial coefficients (x, y) within each abundant Population

# Retrieve stored OLS results and design info
if 'spatial_ols_results_by_pop' not in adata.uns or 'spatial_ols_design_by_pop' not in adata.uns:
    raise ValueError("Required OLS results not found in adata.uns; run the OLS fitting step before this step.")

ols_results_by_pop = adata.uns['spatial_ols_results_by_pop']
design_info_by_pop = adata.uns['spatial_ols_design_by_pop']

spatial_joint_stats_by_pop = {}
pop_summaries = []

for pop, res_df in ols_results_by_pop.items():
    info = design_info_by_pop[pop]
    param_names = info['param_names']
    XtX_inv = np.array(info['XtX_inv'])
    dof_resid = int(info['dof_resid'])

    # Sanity check on design dimensions
    if XtX_inv.shape[0] != XtX_inv.shape[1] or XtX_inv.shape[0] != len(param_names):
        raise ValueError(f"Design information for Population {pop} is inconsistent: XtX_inv shape {XtX_inv.shape}, n_params {len(param_names)}")

    # Indices for x and y in the parameter vector
    if 'x' not in param_names or 'y' not in param_names:
        print(f"Skipping Population {pop}: 'x' or 'y' not in param_names.")
        continue
    ix = param_names.index('x')
    iy = param_names.index('y')

    # Contrast matrix L for [x, y] (2 x p)
    p = len(param_names)
    L = np.zeros((2, p))
    L[0, ix] = 1.0
    L[1, iy] = 1.0

    # Precompute the 2x2 covariance scaling matrix for x,y from XtX_inv
    # For each gene j, cov(beta_xy_j) = sigma2_j * (L @ XtX_inv @ L.T)
    base_cov_xy = L @ XtX_inv @ L.T  # 2 x 2

    # Check conditioning of the 2x2 block
    cond_number = np.linalg.cond(base_cov_xy)
    if cond_number > 1e8:
        print(f"Warning: ill-conditioned spatial covariance block for Population {pop} (cond={cond_number:.2e}); joint tests may be unstable.")

    # Extract per-gene spatial coefficients and residual variances
    required_cols = ['beta_x', 'beta_y', 'sigma2']
    missing_cols = [c for c in required_cols if c not in res_df.columns]
    if missing_cols:
        print(f"Skipping Population {pop}: missing required columns {missing_cols}.")
        continue

    beta_x = res_df['beta_x'].values  # (g,)
    beta_y = res_df['beta_y'].values  # (g,)
    sigma2 = res_df['sigma2'].values  # (g,)
    gene_names = res_df.index.to_numpy()
    n_genes_pop = gene_names.size

    F_vals = np.full(n_genes_pop, np.nan, dtype=float)
    p_vals = np.full(n_genes_pop, np.nan, dtype=float)
    grad_norm = np.sqrt(beta_x**2 + beta_y**2)

    # Invert the common 2x2 base_cov_xy once; per-gene scaling is by sigma2_j
    # For each gene: F = (b^T (sigma2_j * base_cov_xy)^{-1} b) / 2
    #              = (b^T base_cov_xy^{-1} b) / (2 * sigma2_j)
    try:
        base_cov_xy_inv = np.linalg.inv(base_cov_xy)
    except np.linalg.LinAlgError:
        base_cov_xy_inv = np.linalg.pinv(base_cov_xy)

    for j in range(n_genes_pop):
        b = np.array([beta_x[j], beta_y[j]])
        if not np.isfinite(sigma2[j]) or sigma2[j] <= 0:
            continue
        quad = float(b.T @ base_cov_xy_inv @ b)
        F = quad / (2.0 * sigma2[j])
        F_vals[j] = F
        # 2 numerator df, dof_resid denominator df
        p_vals[j] = 1.0 - stats.f.cdf(F, 2, dof_resid)

    # Benjamini-Hochberg FDR within Population (ignore NaNs)
    valid = np.isfinite(p_vals)
    p_valid = p_vals[valid]
    m = p_valid.size
    if m == 0:
        print(f"Population {pop}: no valid p-values for spatial joint tests.")
        continue

    order = np.argsort(p_valid)
    ranks = np.empty_like(order)
    ranks[order] = np.arange(1, m + 1)
    bh_fdr = p_valid * m / ranks
    # enforce monotonicity of BH-adjusted p-values
    bh_fdr_sorted = np.minimum.accumulate(bh_fdr[order][::-1])[::-1]
    bh_fdr[order] = bh_fdr_sorted

    # Put back in original order
    fdr_full = np.full_like(p_vals, np.nan, dtype=float)
    fdr_full[valid] = bh_fdr

    # Store per-gene stats in a DataFrame
    joint_df = pd.DataFrame({
        'gene': gene_names,
        'F_xy': F_vals,
        'pval_xy': p_vals,
        'fdr_xy': fdr_full,
        'beta_x': beta_x,
        'beta_y': beta_y,
        'grad_norm': grad_norm,
        'dof_resid': dof_resid,
    }).set_index('gene')

    spatial_joint_stats_by_pop[pop] = joint_df

    # Population-level summary
    n_genes_total = int(n_genes_pop)
    n_genes_tested = int(m)
    n_sig_005 = int(np.sum((joint_df['fdr_xy'] < 0.05) & np.isfinite(joint_df['fdr_xy'])))
    frac_sig_005 = n_sig_005 / float(n_genes_tested) if n_genes_tested > 0 else np.nan
    median_grad_norm_sig = np.nan
    if n_sig_005 > 0:
        median_grad_norm_sig = float(joint_df.loc[joint_df['fdr_xy'] < 0.05, 'grad_norm'].median())

    pop_summaries.append({
        'Population': pop,
        'n_genes_total': n_genes_total,
        'n_genes_tested': n_genes_tested,
        'n_sig_fdr_0.05': n_sig_005,
        'frac_sig_fdr_0.05': frac_sig_005,
        'median_grad_norm_sig': median_grad_norm_sig,
        'dof_resid': dof_resid
    })

# Store joint-test results and summaries in adata.uns for downstream steps
adata.uns['spatial_joint_stats_by_pop'] = spatial_joint_stats_by_pop
if pop_summaries:
    adata.uns['spatial_joint_pop_summaries'] = pd.DataFrame(pop_summaries).set_index('Population')
else:
    adata.uns['spatial_joint_pop_summaries'] = pd.DataFrame(columns=[
        'n_genes_total', 'n_genes_tested', 'n_sig_fdr_0.05',
        'frac_sig_fdr_0.05', 'median_grad_norm_sig', 'dof_resid'])

# Brief textual summary: rank Populations by fraction of spatially significant genes
summary_df = adata.uns['spatial_joint_pop_summaries'].sort_values('frac_sig_fdr_0.05', ascending=False)
print("Population-level summary of spatial joint tests (sorted by fraction of FDR<0.05 genes):")
print(summary_df[['n_genes_total', 'n_genes_tested', 'n_sig_fdr_0.05', 'frac_sig_fdr_0.05', 'median_grad_norm_sig', 'dof_resid']])

Population-level summary of spatial joint tests (sorted by fraction of FDR<0.05 genes):
            n_genes_total  n_genes_tested  n_sig_fdr_0.05  frac_sig_fdr_0.05  \
Population                                                                     
PF                    238             238             224           0.941176   
PB                    238             238             216           0.907563   
PK                    238             238             207           0.869748   
PH                    238             238             206           0.865546   
PA                    238             238             204           0.857143   
PC                    238             238             203           0.852941   
PD                    238             238             201           0.844538   
PG                    238             238             191           0.802521   
PE                    238             238             190           0.798319   
PN                    238       

### Agent Interpretation

These results strongly support the hypothesis that many genes have significant spatial gradients within Populations even after adjusting for Complexity, Purity, and Sample_ID, and that the strength of this effect varies substantially by Population.

Key points and how they inform next steps:

1. **Prevalence of spatially structured genes is extremely high.**  
   - Most Populations have >70% of tested genes with FDR<0.05 for the joint x/y test; PF, PB, PK, PH, PA, PC, PD are all >0.84, with PF at ~94%.  
   - Even the “lowest” Populations (PV, PAA, PX, PY, PZ, PW) still have ~0.42–0.60 of genes significant.  
   - This indicates spatial gradients are pervasive after controlling for Complexity, Purity, and Sample_ID; the null that β_x = β_y = 0 is rejected for the majority of genes.

2. **Variation across Populations is nontrivial, consistent with the hypothesis.**  
   - The fraction of spatially significant genes ranges from ~0.42 to ~0.94.  
   - This spread is precisely the type of between-Population heterogeneity the hypothesis calls for (Populations with “unusually strong spatial structure”).  
   - Right now, high-FDR fractions cluster among PF–PL, low fractions at PW–PV. This gives a natural ranking to carry into steps 2–4.

3. **Effect sizes are small in absolute units but heterogeneous.**  
   - Median gradient norms for significant genes span an order of magnitude (e.g., ~3–6×10⁻⁵ for many high-fraction Populations versus ~2–9×10⁻⁴ for PW, PX, PAA, PV).  
   - Since coordinates were only centered (not rescaled), the absolute gradient_norm is hard to interpret biologically, but the **relative differences** between Populations (e.g., PF vs PW) are informative.  
   - Populations like **PW, PX, PAA, PV** have lower fractions of significant genes but larger median gradient norms among those that are significant, suggesting more “sparse but strong” spatial structure, in contrast to “dense but shallow” gradients in PF–PL. That’s an interesting pattern to keep in mind for interpretation later.

4. **Statistical robustness is good but needs a few checks.**  
   - dof_resid is large for most Populations (thousands to tens of thousands), so F-tests will be extremely powered. That partly explains the high fraction of significant genes—even tiny effects will be detected.  
   - Because XtX_inv is shared within a Population and you explicitly checked conditioning, the joint tests should be numerically stable overall; but it would be worth briefly inspecting which Populations emitted the ill-conditioning warnings, if any, to avoid overinterpreting their extremes.  
   - Within-population BH is appropriate for the current goal (comparing fractions within each Population). If you later want to talk about “global” rates across Populations, you’ll need to layer another control.

How to leverage this for the remaining steps:

**1. Prioritize Populations for step 2 (Complexity ~ Purity + Sample_ID (+ x,y)):**

- For **“high global spatial structure”** Populations, pick those with the highest `frac_sig_fdr_0.05`, e.g.:  
  - PF, PB, PK, PH, PA, PC, PD, PG, PE, PN, PU, PI (say top ~8–10 as a starting set).  
- For **contrast**, deliberately include a few lower-fraction Populations (e.g. PX, PY, PW, PZ, PAA, PV) to see whether Complex­ity’s spatial dependence is indeed weaker there, or whether they simply have fewer “modest” spatial genes but some strong outliers.

In step 2, the main question is:  
> Does adding x,y materially raise R² for Complexity after controlling for Purity and Sample_ID?

Given the extreme power, you should not focus solely on F-test p-values (which will likely be tiny for many Populations). Instead:

- Compute **ΔR² = R²(model with x,y) − R²(model without x,y)** and focus on its magnitude.  
- Rank Populations by ΔR² and treat those in, say, the top quartile or top 5–8 as “Complexity is strongly spatial” Populations.

It’s plausible that:
- Some high-fraction Populations (e.g., PF, PB) also show large ΔR² → clear evidence that intra-Population maturation (Complexity) is spatially organized.  
- Some “sparse-but-strong” Populations (PW, PX, PAA, PV) might have **smaller** ΔR² if only a subset of genes/regions drive spatial structure unrelated to Complexity, which would be equally interesting for interpretation (e.g., spatially specialized subregions rather than a pure maturation gradient).

**2. Anticipate and design step 3 (defining robust spatial-gradient genes) based on current results:**

You already have for each Population:

- `F_xy`, `pval_xy`, `fdr_xy`, `beta_x`, `beta_y`, `grad_norm` per gene.

For those Populations that, in step 2, show **substantial ΔR² for Complexity**:

- Define a stricter set of **robust spatial-gradient genes** using both:
  - FDR threshold (e.g., fdr_xy < 0.05 or more stringent < 0.01 to avoid saturating the list), and  
  - A **gradient_norm threshold** (e.g., choose a quantile within that Population’s significant set, or an absolute threshold like gradient_norm above the median or 75th percentile).  
- Because median gradient_norm differs across Populations, using a **within-Population quantile** of gradient_norm to define “large” is more defensible than any global numeric cutoff.

This will help align step 3 to the hypothesis: we care about genes whose spatial gradients are both statistically robust and **biologically non-negligible**, not just tiny but detectable trends.

**3. Plan for sample-stratified consistency checks (step 3) using current structure:**

You’ve controlled for Sample_ID in the global fits; in step 3 you’ll:

- Refit per-sample models within a given Population, with x,y and Complexity/Purity included as in the parent model.  
- For the robust-gene set, examine:
  - Sign concordance of β_x, β_y across samples (e.g., proportion of samples with same sign for each coefficient; maybe also require nontrivial magnitude).  
  - Possibly, the stability of gradient_norm ranks across samples.

Given how many genes are significant, you’ll likely want to **pre-filter** robust genes by FDR + gradient_norm as above before doing per-sample refits to keep things manageable and highlight only the clearest patterns.

**4. Implications for the hypothesis so far:**

- The existence of many FDR-significant x/y terms per Population, with clear between-Population differences in their prevalence, is already **strong partial validation** of the hypothesis:
  - Spatial gradients persist after adjusting for Complexity, Purity, and Sample_ID.
  - Populations differ in the fraction and effect-size distribution of these gradients.

What remains to directly support the “maturation” interpretation is:

- Showing that **Complexity itself gains nontrivial explanatory power from spatial coordinates** within certain Populations (step 2).  
- Demonstrating that the **robust spatial-gradient genes are coherent across samples** (step 3), arguing against sample-specific artifacts and towards genuine intra-Population maturation domains.

**5. Additional checks and small improvements you might consider:**

- **Inspect edge Populations** (very high or very low `frac_sig_fdr_0.05`) for:
  - Warnings about ill-conditioned spatial covariance blocks (already printed if present).  
  - Odd dof_resid values (none look pathological here, but note that low dof_resid Populations like PAA, PW, PZ may have less stable per-sample fits in step 3).
- Consider summarizing for each Population:
  - The **distribution** (not just median) of gradient_norm among significant genes (e.g., IQR, 90th percentile) to better separate “many mild” vs “few strong” patterns, which may guide how you narrate differences in step 4’s report.

In summary: this step has worked as intended and provides a solid quantitative basis for the next stages. I’d move on to:

1) Fit the nested Complexity models across Populations and rank by ΔR²;  
2) Use the existing joint-test outputs to define robust spatial-gradient genes in those high-ΔR² Populations;  
3) Then do the per-sample consistency and final interpretive report tying specific Populations to clear intra-Population, spatially organized maturation patterns.

## Next Steps
Step 1: Using the existing per-Population OLS fits and joint x/y F-tests, fit two nested linear models per sufficiently abundant Population with Complexity as the outcome and predictors (i) Purity + Sample_ID and (ii) Purity + Sample_ID + x + y (using the same centered covariates and Sample_ID coding as in the expression models); compute R² for both models, ΔR², and nested-model F-test statistics and p-values, and produce a text summary ranking Populations by ΔR² and reporting which Populations show statistically and practically meaningful spatial organization of Complexity.
Step 2: For Populations with clearly non-negligible ΔR² (e.g., above a chosen quantile or absolute threshold), define robust spatial-gradient genes by combining the existing FDR-significant joint x/y tests (e.g., fdr_xy < 0.05) with a within-Population gradient_norm cutoff (such as ≥ median gradient_norm among significant genes), then, within each such Population and for each robust gene, refit the linear model separately within each Sample_ID (dropping Sample_ID fixed effects) to obtain per-sample β_x and β_y; summarize per-gene cross-sample sign concordance and variability of gradient_norm, and print a ranked text table of robust genes per Population with their spatial statistics and concordance metrics.
Step 3: Generate a final text-only report that (a) lists Populations ordered by ΔR² and by fraction of FDR-significant spatial genes, (b) for each high-ΔR² Population, prints the number of robust spatial-gradient genes, their typical gradient_norm and cross-sample concordance, and a few example genes with detailed statistics, and (c) interprets which Populations show the clearest evidence that intra-population maturation states (Complexity) are spatially organized beyond what is captured by global Complexity and Purity, emphasizing differences between Populations with many mild gradients versus those with fewer but stronger gradients.

## Fit, within each sufficiently abundant Population, two nested OLS models for Complexity with and without spatial coordinates (x,y) after adjusting for Purity and Sample_ID, compute ΔR² and a nested-model F-test for the added spatial terms, apply BH FDR across Populations, and store plus print a text summary of which Populations show spatially structured Complexity.

In [ ]:
import numpy as np
import pandas as pd
from scipy import linalg, stats

# Step: Model Complexity as a function of spatial position and covariates within each Population

# We reuse spatial coordinates and covariates from adata.obs, as in the OLS expression models
obs = adata.obs.copy()
spatial = adata.obsm['spatial']
obs['x'] = spatial[:, 0]
obs['y'] = spatial[:, 1]

# Ensure categorical types
if not pd.api.types.is_categorical_dtype(obs['Populations']):
    obs['Populations'] = obs['Populations'].astype('category')
if not pd.api.types.is_categorical_dtype(obs['Sample_ID']):
    obs['Sample_ID'] = obs['Sample_ID'].astype('category')

# Use the same abundance threshold as before
min_cells = 200
pop_counts = obs['Populations'].value_counts()
abundant_pops = pop_counts[pop_counts >= min_cells].index.tolist()

complexity_results = []

for pop in abundant_pops:
    pop_mask = (obs['Populations'] == pop).values
    n_pop = int(pop_mask.sum())
    if n_pop < min_cells:
        continue

    obs_pop = obs.loc[pop_mask, :].copy()

    # Center numeric covariates within Population, matching the expression-model design
    for col in ['x', 'y', 'Purity']:
        obs_pop[col] = obs_pop[col].astype(float)
        obs_pop[col] = obs_pop[col] - obs_pop[col].mean()

    # Response: Complexity (do not center for R^2 comparability, but centering would not change R^2)
    y_resp = obs_pop['Complexity'].astype(float).values

    # Design matrix for reduced model: intercept + Purity + Sample_ID fixed effects
    sample_dummies = pd.get_dummies(obs_pop['Sample_ID'], drop_first=True)
    design_reduced = pd.DataFrame({'intercept': np.ones(n_pop), 'Purity': obs_pop['Purity'].values})
    if sample_dummies.shape[1] > 0:
        design_reduced = pd.concat([design_reduced, sample_dummies.reset_index(drop=True)], axis=1)

    X_red = design_reduced.values.astype(float)
    n_params_red = X_red.shape[1]
    dof_red = n_pop - n_params_red
    if dof_red <= 0:
        # Skip pathological designs
        continue

    # Fit reduced model by OLS
    XtX_red = X_red.T @ X_red
    try:
        XtX_red_inv = linalg.inv(XtX_red)
    except linalg.LinAlgError:
        XtX_red_inv = linalg.pinv(XtX_red)
    beta_red = XtX_red_inv @ (X_red.T @ y_resp)
    y_hat_red = X_red @ beta_red
    resid_red = y_resp - y_hat_red
    ss_res_red = float(np.sum(resid_red ** 2))
    ss_tot = float(np.sum((y_resp - y_resp.mean()) ** 2))
    r2_red = 1.0 - ss_res_red / ss_tot if ss_tot > 0 else np.nan

    # Design matrix for full model: intercept + Purity + x + y + Sample_ID fixed effects
    design_full = pd.DataFrame({
        'intercept': np.ones(n_pop),
        'Purity': obs_pop['Purity'].values,
        'x': obs_pop['x'].values,
        'y': obs_pop['y'].values,
    })
    if sample_dummies.shape[1] > 0:
        design_full = pd.concat([design_full, sample_dummies.reset_index(drop=True)], axis=1)

    X_full = design_full.values.astype(float)
    n_params_full = X_full.shape[1]
    dof_full = n_pop - n_params_full
    if dof_full <= 0 or n_params_full <= n_params_red:
        # Skip if full model is not properly nested or has no extra df
        continue

    # Fit full model by OLS
    XtX_full = X_full.T @ X_full
    try:
        XtX_full_inv = linalg.inv(XtX_full)
    except linalg.LinAlgError:
        XtX_full_inv = linalg.pinv(XtX_full)
    beta_full = XtX_full_inv @ (X_full.T @ y_resp)
    y_hat_full = X_full @ beta_full
    resid_full = y_resp - y_hat_full
    ss_res_full = float(np.sum(resid_full ** 2))
    r2_full = 1.0 - ss_res_full / ss_tot if ss_tot > 0 else np.nan

    # Nested-model F-test for adding x,y
    df_num = n_params_full - n_params_red  # should be 2
    df_den = dof_full
    if df_num <= 0 or df_den <= 0:
        continue
    ms_num = (ss_res_red - ss_res_full) / df_num
    ms_den = ss_res_full / df_den
    if ms_den <= 0:
        F_stat = np.nan
        p_val = np.nan
    else:
        F_stat = ms_num / ms_den
        p_val = 1.0 - stats.f.cdf(F_stat, df_num, df_den)

    delta_r2 = r2_full - r2_red

    complexity_results.append({
        'Population': pop,
        'n_cells': n_pop,
        'r2_reduced': r2_red,
        'r2_full': r2_full,
        'delta_r2': delta_r2,
        'F_stat_xy': F_stat,
        'pval_xy': p_val,
        'df_num': df_num,
        'df_den': df_den
    })

# Compile results into a DataFrame
if complexity_results:
    complexity_df = pd.DataFrame(complexity_results).set_index('Population')
else:
    complexity_df = pd.DataFrame(columns=['n_cells', 'r2_reduced', 'r2_full', 'delta_r2', 'F_stat_xy', 'pval_xy', 'df_num', 'df_den'])

# Benjamini–Hochberg FDR across Populations for the Complexity spatial F-test
if not complexity_df.empty:
    pvals = complexity_df['pval_xy'].values
    valid = np.isfinite(pvals)
    p_valid = pvals[valid]
    m = p_valid.size
    if m > 0:
        order = np.argsort(p_valid)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, m + 1)
        bh = p_valid * m / ranks
        bh_sorted = np.minimum.accumulate(bh[order][::-1])[::-1]
        bh[order] = bh_sorted
        fdr_full = np.full_like(pvals, np.nan, dtype=float)
        fdr_full[valid] = bh
        complexity_df['fdr_xy'] = fdr_full
    else:
        complexity_df['fdr_xy'] = np.nan
else:
    complexity_df['fdr_xy'] = np.nan

# Store results
adata.uns['complexity_spatial_models'] = complexity_df

# Print a textual summary ranked by delta_r2
if not complexity_df.empty:
    summary = complexity_df.sort_values('delta_r2', ascending=False)
    print("Nested models for Complexity (ranked by ΔR² = R²_full − R²_reduced):")
    print(summary[['n_cells', 'r2_reduced', 'r2_full', 'delta_r2', 'F_stat_xy', 'pval_xy', 'fdr_xy', 'df_num', 'df_den']])
else:
    print("No sufficiently abundant Populations for Complexity spatial modeling.")

Nested models for Complexity (ranked by ΔR² = R²_full − R²_reduced):
            n_cells  r2_reduced   r2_full  delta_r2    F_stat_xy  \
Population                                                         
PW             2027    0.200169  0.358707  0.158538   249.811762   
PD            16624    0.074327  0.227974  0.153646  1653.629651   
PE            16511    0.045682  0.198954  0.153272  1579.027028   
PQ             5429    0.199353  0.350469  0.151117   630.844476   
PA            30380    0.208130  0.355878  0.147747  3483.559459   
PC            17584    0.113410  0.257805  0.144396  1709.921939   
PJ             9488    0.029648  0.156905  0.127256   715.604016   
PP             5458    0.120337  0.246160  0.125823   454.995608   
PO             5845    0.130088  0.247151  0.117063   453.963197   
PAA            1027    0.256596  0.364235  0.107639    86.431008   
PL             8052    0.068239  0.175515  0.107276   523.445974   
PI            10441    0.291861  0.397855  0.10

### Agent Interpretation

These results are very supportive of the core hypothesis that, in many cardiac Populations, Complexity itself is spatially structured even after adjusting for Purity and Sample_ID. A few specific points and how they should guide the next steps:

1. **Strength and prevalence of Complexity–space coupling**

   - Many Populations show **substantial ΔR² (~0.10–0.16)** when adding x,y:
     - Top group (ΔR² ≥ ~0.14–0.16): PW, PD, PE, PQ, PA, PC.
     - Next group (ΔR² ~0.10–0.13): PJ, PP, PO, PAA, PL, PI, PV, PB, PK, PH, PX.
   - These ΔR² values are *not trivial*: x,y explain an additional ~10–16% of Complexity variance beyond Purity and inter-sample shifts, which is a strong effect for a single linear gradient in heterogeneous tissue.
   - F-statistics for x,y are huge with essentially zero FDR for almost all Populations (fdr_xy ≈ 1.4e−16), so **statistical evidence is overwhelming** that Complexity has spatial gradients in nearly every well-powered Population.

   Interpretation:
   - The hypothesis that “spatial organization of Complexity is widespread” is clearly supported.
   - The variation in ΔR² across Populations already gives you a first **ranking of how tightly Complexity is spatially organized** within each Population.

2. **Populations with clearest maturation-like spatial organization**

   If we interpret larger ΔR² as “Complexity is more strongly organized in space,” then:

   - **High-ΔR², high-n Populations** that should be prioritized:
     - PD, PE, PA, PC (n ≈ 16k–30k; ΔR² ~0.14–0.15).
     - PQ, PW (ΔR² ~0.15–0.16).
   - **Moderately strong** but still very interesting:
     - PB (ΔR² ~0.10, r2_full ≈ 0.58), PG (ΔR² ~0.08, r2_full ≈ 0.52), PM (ΔR² ~0.02 but very high baseline r² with Purity/sample), PX, PI, PV, etc.
   - **Low-ΔR² but still statistically significant**:
     - PS, PT, PU, PZ, PR, PF, PN, PY.

   Differences in **r2_reduced vs r2_full** are also informative:
   - Populations like **PB, PG, PM, PR** already have very high r2_reduced (≥0.43–0.75), suggesting Complexity is strongly explained by Purity and sample-level effects even before adding x,y. For these, spatial gradients in Complexity exist but are more incremental.
   - Populations like **PD, PE, PJ, PK, PL, PH** have low r2_reduced but large ΔR², meaning **spatial gradients account for a large new share of Complexity variation that Purity and Sample_ID do not capture**. These are particularly good candidates for “maturation patterns not captured by Complexity alone” in a global sense, because Complexity’s *residual* structure is spatial.

3. **How to choose the “non-negligible ΔR²” set for the next step**

   You’ll now define “high-ΔR² Populations” for robust gradient-gene analysis. A defensible strategy using these results:

   - Use an **absolute ΔR² cutoff** plus maybe a quantile rule.
   - For example:
     - Strongly spatial: ΔR² ≥ 0.10 → {PW, PD, PE, PQ, PA, PC, PJ, PP, PO, PAA, PL, PI, PV, PB, PK, PH, PX}
     - Optionally define an even more stringent subset for deep inspection later: ΔR² ≥ 0.14 → {PW, PD, PE, PQ, PA, PC}.
   - You might also track populations where **ΔR² is modest but r2_full is very high** (PB, PG, PM, PR), as they might represent “more homogeneous but still spatially tuned” maturation states.

   For biological interpretation later, it will help to categorize Populations into:
   - “Low baseline, big spatial gain” (e.g., PD, PE, PJ, PK, PL, PH).
   - “High baseline, moderate spatial gain” (e.g., PB, PG, PM, PR).
   This matches the planned contrast between Populations with many mild gradients vs fewer, stronger, but already-explained patterns.

4. **Implications for the gene-level spatial gradient analysis**

   The next step is to define robust spatial-gradient genes by combining:
   - FDR-significant joint x/y expression tests, and
   - A gradient_norm cutoff (e.g., ≥ median among significant genes).

   Based on these Complexity results:

   - In **high-ΔR² Populations**, you should expect:
     - A non-trivial fraction of genes whose spatial gradients in expression run parallel or anti-parallel to the Complexity gradient.
     - Possibly more coherent gradient directions across samples (higher sign concordance in β_x, β_y) than in low-ΔR² Populations.
   - In **low-ΔR² but FDR-significant Populations** (e.g., PT, PU, PZ, PR), spatial patterns of expression may exist but might be:
     - Weaker in magnitude (lower gradient_norm).
     - Less closely aligned with Complexity (more “idiosyncratic” spatial gene gradients).

   Specific recommendations for the next step:
   - **Stratify all downstream summaries by the ΔR² tier** of the Population. For each Pop:
     - Count robust spatial genes.
     - Summarize median/quantile gradient_norm.
     - Report cross-sample sign concordance of (β_x, β_y).
   - **Compare strong vs weak Complexity-gradient Populations**:
     - Do high-ΔR² Populations have a larger fraction of robust spatial genes that are also strongly Complexity-associated (from previous analyses)?
     - Are gradient directions for Complexity-aligned genes more consistent across Sample_ID in high-ΔR² Populations?

5. **Checks and small refinements before the gene-level step**

   - You’re already centering x,y and Purity within Population, and including Sample_ID fixed effects, which is consistent with the expression models; that’s good for comparability.
   - It could be informative (not required, but helpful) to:
     - Store the **β_x and β_y coefficients for Complexity** per Population (from `beta_full`) to:
       - Project gene-level gradients onto the Complexity gradient later (e.g., via cosine similarity between gene gradient vectors and the Complexity gradient vector).
       - Directly test if robust spatial-gradient genes tend to align with the Complexity gradient direction.
   - Consider computing a simple **effect-size ratio** like ΔR² / (1 − r2_reduced) to quantify “how much of the previously unexplained Complexity variance is captured by space” for each Population; this may sharpen the high/low ΔR² grouping.

6. **How these results speak to the hypothesis right now**

   - The **“widespread” part** of the hypothesis is supported: essentially every sufficiently abundant Population shows statistically strong spatial organization of Complexity (very low FDR for x,y).
   - The **“vary in strength across Populations”** part is clearly visible in the spread of ΔR² (~0.004 to ~0.16).
   - The remaining, more specific claim—**that in a subset of Populations, spatial Complexity gradients are tightly linked to the spatial organization of expression-space gradients (i.e., maturation patterns not captured by Complexity alone)**—will be tested directly in the next steps by:
     - Quantifying, per Population, whether robust spatial gradient genes are common and coherent.
     - Assessing cross-sample gradient sign concordance and strength.

   So far, the data argue that such subsets are likely to exist, especially among the high-ΔR² Populations (PD, PE, PA, PC, PQ, PW, and perhaps PB, PG, PM, PR as a distinct pattern). The planned per-gene, per-sample re-fitting and concordance analyses are exactly what’s needed to confirm whether those Populations harbor clear, spatially organized intra-population maturation programs beyond what global Complexity and Purity capture.

In summary, proceed to the next step focusing on ΔR²-stratified Populations, emphasizing the high-ΔR² group as candidates where Complexity gradients and gene-level spatial gradients are most likely to be tightly coupled and biologically interpretable.

## Next Steps
Step 1: For each sufficiently abundant Population, use the previously computed Complexity spatial models (stored in adata.uns['complexity_spatial_models']) to select a high-ΔR² subset (e.g., Populations with ΔR² ≥ 0.10), then, within each selected Population, define robust spatial-gradient genes by intersecting the set of genes with significant joint spatial expression effects (fdr_xy < 0.05 in adata.uns['spatial_joint_stats_by_pop'][pop]) with those whose gradient_norm is at or above the median gradient_norm among that Population’s significant genes, and compile a per-Population table of these robust genes including β_x, β_y, grad_norm, and FDR.
Step 2: For each high-ΔR² Population and each of its robust spatial-gradient genes, refit the linear model separately within each Sample_ID (cells restricted to that Population and sample; predictors: centered x, y, Complexity, Purity with no Sample_ID fixed effects), extract per-sample β_x and β_y, and summarize for each gene its cross-sample sign concordance for x and y, median and IQR of gradient_norm across samples, and the fraction of samples in which the joint spatial test (2-df F-test) remains nominally significant (p < 0.05); finally, print a ranked, text-only table per Population listing robust genes ordered by a composite metric combining concordance, typical gradient strength, and nominal per-sample significance, alongside each Population’s ΔR² and baseline fraction of spatially significant genes to highlight where spatially organized maturation-like gradients are most reproducible.

## This code identifies high-ΔR² Populations using previously fit Complexity~(Purity,Sample_ID,x,y) models, then, within each such Population, defines robust spatial-gradient genes as those with FDR-significant joint x/y effects and gradient norms at or above the median among significant genes, storing these gene sets and a textual summary in adata.uns for use in the final per-sample consistency analysis.

In [ ]:
import numpy as np
import pandas as pd

# Step: define robust spatial-gradient genes in high-ΔR² Populations

# Retrieve required objects
complexity_df = adata.uns.get('complexity_spatial_models', None)
joint_by_pop = adata.uns.get('spatial_joint_stats_by_pop', None)

if complexity_df is None or joint_by_pop is None:
    raise ValueError("Required results not found in adata.uns: need 'complexity_spatial_models' and 'spatial_joint_stats_by_pop'.")

# Identify high-ΔR² Populations (here: absolute cutoff ΔR² ≥ 0.10)
high_delta_threshold = 0.10
high_delta_pops = complexity_df.index[complexity_df['delta_r2'] >= high_delta_threshold].tolist()

if not high_delta_pops:
    print(f"No Populations with delta_r2 ≥ {high_delta_threshold}; relaxing threshold to top quartile.")
    # Fallback: take top quartile of Populations by delta_r2
    if not complexity_df.empty:
        q75 = complexity_df['delta_r2'].quantile(0.75)
        print(f"Using relaxed high-ΔR² threshold at 75th percentile: {q75:.4f}")
        high_delta_pops = complexity_df.index[complexity_df['delta_r2'] >= q75].tolist()

print("High-ΔR² Populations selected for robust-gradient analysis:")
print(high_delta_pops)

robust_genes_by_pop = {}

for pop in high_delta_pops:
    if pop not in joint_by_pop:
        print(f"Skipping Population {pop}: no spatial joint stats found.")
        continue
    joint_df = joint_by_pop[pop].copy()

    # Require valid FDR values
    valid = np.isfinite(joint_df['fdr_xy'])
    joint_df = joint_df.loc[valid]
    if joint_df.empty:
        print(f"Population {pop}: no valid spatial joint-test results.")
        continue

    # Significant genes by FDR
    sig_mask = joint_df['fdr_xy'] < 0.05
    sig_df = joint_df.loc[sig_mask]
    n_sig = sig_df.shape[0]
    n_total = joint_df.shape[0]
    if n_sig == 0:
        print(f"Population {pop}: no genes with fdr_xy < 0.05.")
        robust_genes_by_pop[pop] = pd.DataFrame(columns=['beta_x', 'beta_y', 'grad_norm', 'fdr_xy'])
        continue

    # Within-Population gradient_norm cutoff: median among significant genes
    median_grad = float(sig_df['grad_norm'].median())
    robust_mask = (sig_df['grad_norm'] >= median_grad)
    robust_df = sig_df.loc[robust_mask].copy()

    # Keep only relevant columns and sort by gradient strength
    robust_df = robust_df[['beta_x', 'beta_y', 'grad_norm', 'fdr_xy']]
    robust_df.sort_values('grad_norm', ascending=False, inplace=True)

    robust_genes_by_pop[pop] = robust_df

    print(f"Population {pop}: {n_total} genes tested, {n_sig} with fdr_xy<0.05; {robust_df.shape[0]} robust genes (grad_norm ≥ median among significant).")

# Store robust gene sets for use in the next step
adata.uns['robust_spatial_gradient_genes_by_pop'] = robust_genes_by_pop

# Brief textual summary table across Populations
summary_rows = []
for pop in high_delta_pops:
    robust_df = robust_genes_by_pop.get(pop, pd.DataFrame())
    n_robust = robust_df.shape[0]
    n_sig = np.nan
    n_total = np.nan
    if pop in joint_by_pop:
        jdf = joint_by_pop[pop]
        valid = np.isfinite(jdf['fdr_xy'])
        n_total = int(valid.sum())
        n_sig = int(((jdf['fdr_xy'] < 0.05) & valid).sum())
    delta_r2 = float(complexity_df.loc[pop, 'delta_r2']) if pop in complexity_df.index else np.nan
    frac_sig = (n_sig / n_total) if (isinstance(n_total, (int, np.integer)) and n_total > 0 and n_sig is not None) else np.nan
    summary_rows.append({
        'Population': pop,
        'delta_r2': delta_r2,
        'n_genes_total': n_total,
        'n_sig_fdr_0.05': n_sig,
        'frac_sig_fdr_0.05': frac_sig,
        'n_robust_genes': n_robust
    })

if summary_rows:
    summary_df = pd.DataFrame(summary_rows).set_index('Population').sort_values('delta_r2', ascending=False)
    print("\nSummary of high-ΔR² Populations and robust spatial-gradient genes:")
    print(summary_df[['delta_r2', 'n_genes_total', 'n_sig_fdr_0.05', 'frac_sig_fdr_0.05', 'n_robust_genes']])
else:
    print("No high-ΔR² Populations or robust genes identified.")


High-ΔR² Populations selected for robust-gradient analysis:
['PA', 'PB', 'PC', 'PD', 'PE', 'PH', 'PI', 'PJ', 'PK', 'PL', 'PO', 'PP', 'PQ', 'PV', 'PW', 'PAA']
Population PA: 238 genes tested, 204 with fdr_xy<0.05; 102 robust genes (grad_norm ≥ median among significant).
Population PB: 238 genes tested, 216 with fdr_xy<0.05; 108 robust genes (grad_norm ≥ median among significant).
Population PC: 238 genes tested, 203 with fdr_xy<0.05; 102 robust genes (grad_norm ≥ median among significant).
Population PD: 238 genes tested, 201 with fdr_xy<0.05; 101 robust genes (grad_norm ≥ median among significant).
Population PE: 238 genes tested, 190 with fdr_xy<0.05; 95 robust genes (grad_norm ≥ median among significant).
Population PH: 238 genes tested, 206 with fdr_xy<0.05; 103 robust genes (grad_norm ≥ median among significant).
Population PI: 238 genes tested, 179 with fdr_xy<0.05; 90 robust genes (grad_norm ≥ median among significant).
Population PJ: 238 genes tested, 162 with fdr_xy<0.05; 81 ro

### Agent Interpretation

These results are very encouraging for the hypothesis and set you up well for the next refitting / cross-sample step.

Key points:

1. **High-ΔR², well-powered Populations are abundant.**  
   - You have 16 Populations with ΔR² ≥ ~0.10, with PW, PD, PE, PQ, PA, PC, PB, PK, PH all around 0.10–0.16.  
   - This means spatial coordinates explain a substantial additional fraction of Complexity beyond non-spatial covariates, exactly the “spatially organized” regimes your hypothesis targets.

2. **Spatial effects are widespread, not sparse.**  
   - In many of these Populations, ~70–90% of genes tested are spatially significant (FDR_xy < 0.05).  
     - PA: 204/238 (0.86),  
     - PB: 216/238 (0.91),  
     - PC: 203/238 (0.85),  
     - PK: 207/238 (0.87),  
     - PH: 206/238 (0.87), etc.  
   - Even at the lower end (PAA, PV), you still have ~40% of genes with significant spatial effects.  
   - This suggests that within these Populations, “spatial structure” is not limited to a handful of markers but is a pervasive axis of variation, which is compatible with maturation-like gradients rather than a single discrete subcluster.

3. **You have sizeable robust-gradient gene sets per Population.**  
   - By thresholding at the median gradient_norm among significant genes, you retain 50–108 genes per Population (roughly half of the significant set).  
   - For the strongest-ΔR² Populations:
     - PW (ΔR² 0.1585): 71 robust genes  
     - PD (0.1536): 101 robust  
     - PE (0.1533): 95 robust  
     - PA/PC/PB/PK/PH: ~100+ robust each  
   - This is large enough for meaningful downstream ranking, cross-sample statistics, and any later pathway / module-style reasoning, but selective enough to focus on strong gradients.

4. **Methodological choices look reasonable and distinct from past analyses.**  
   - You are explicitly using **joint x,y spatial effects** (not binned high/low Complexity tertiles as in Analysis 1 and not neighborhood composition as in Analysis 2).  
   - The robust-definition (FDR_xy < 0.05 + gradient_norm ≥ median) makes your “candidate maturation-gradient genes” clearly stronger than average spatial genes within each Population, without using arbitrary absolute cutoffs that would vary with measurement scale.

How this informs the hypothesis:

- The hypothesis has two main components:
  1. **There exist intra-population genes with robust, non-negligible spatial gradients** in Populations where Complexity is spatially structured (large ΔR²).  
     - Your current step directly supports this: in every high-ΔR² Population, you find dozens of robust-gradient genes, and the fraction of all genes with spatial effects is high. This is good evidence that there are strong spatial expression gradients within these Populations beyond global Complexity/Purity alone.
  2. **The directions of these gradients are consistent across samples.**  
     - This part is **not yet tested**. Your current results just pool across samples; they don’t say whether β_x/β_y retain the same sign and similar orientation per sample. That will be decided entirely by the next step (per-sample refits).

Suggestions for the next steps / refinements:

1. **Prioritize Populations for deeper inspection.**  
   For the per-sample refitting and ranking, it may be worth:
   - Focusing first on Populations with both high ΔR² and a high fraction of spatially significant genes, e.g.:  
     - PW, PD, PE, PQ, PA, PC, PB, PK, PH.  
   - You can still run the full pipeline on all 16, but for manual interpretation later (e.g., spatial maps of a few genes), these will likely be the most compelling.

2. **Per-sample model fitting: pay attention to power and sample coverage.**  
   - Some Populations may be missing in a subset of samples or have very few cells in certain samples. For each Population–Sample_ID pair, record:
     - Number of cells used in the fit  
     - Whether the model converged and whether the 2-df spatial test is valid  
   - In your summary stats (sign concordance, fraction of samples with p < 0.05), restrict to samples with sufficient n (e.g., ≥30–50 cells per Population per sample) so that “discordant” signs aren’t driven by noise.

3. **Careful definition of the composite ranking metric.**  
   To identify the most compelling maturation-like candidates, consider a ranking per Population combining:
   - **Sign-concordance for x and y**:
     - e.g., fraction of samples where sign(β_x) matches the majority sign, and similarly for β_y.  
     - You might also encode direction as an **angle** (arctan2(β_y, β_x)) and assess circular dispersion across samples; but as a first pass, sign-concordance with an angle-consistency check for top candidates is fine.
   - **Cross-sample median gradient_norm and IQR**:  
     - High median, low IQR = strong, reproducible gradient.  
   - **Fraction of samples with nominal spatial significance (p_xy < 0.05)**:
     - To avoid penalizing genes in Populations absent in some samples, compute this fraction over samples where the Population is present and adequately powered.
   Consider, for example, a simple composite like:
   - Score = (sign_concordance_x + sign_concordance_y)/2  
     × median_grad_norm  
     × fraction_sig_xy  
   Then rank by Score within each Population.

4. **Interpretation checks once the per-sample results are in.**  
   After the next step, for top-ranked genes per Population:
   - Plot β_x vs β_y across samples (per gene) to ensure gradients cluster around a common direction vector rather than splitting into opposing modes.  
   - For 2–3 of the best-scoring genes in 2–3 of the top Populations, generate spatial maps (expression and/or residuals after adjusting for Complexity/Purity) to visually inspect whether:
     - There is a smooth gradient rather than sharp, discrete subdomains.  
     - Gradient direction is consistent across samples with similar anatomical orientation (you’re not allowed external labels, but you can qualitatively check similarity).

5. **Guard against confounding with Complexity and Purity.**  
   - The design in the next step already includes Complexity and Purity as covariates, which is good.  
   - Once you have per-sample β_x/β_y, verify that your top genes’ spatial gradients aren’t trivially mirroring the previously fitted Complexity gradient:
     - Compute per-sample correlation between each gene’s β-vector (β_x, β_y) and the Complexity β-vector for that Population–sample combination.  
     - Genes whose gradient vectors are not highly colinear with Complexity’s gradient but still show high concordance and significance are especially interesting as “independent maturation axes” (e.g., regionally biased maturation within a Population).

6. **Consider heterogeneity across Populations.**  
   Once you’ve identified top robust genes per Population:
   - Check whether some genes appear as robust, directionally concordant gradients in **multiple Populations**, potentially indicating a shared maturation program expressed in different cell types.  
   - Conversely, Populations with high ΔR² but relatively lower frac_sig_fdr_0.05 or fewer robust genes (e.g., PAA, PV) may represent more discrete spatial niches or more heterogeneous mixtures where a single dominant maturation axis is weaker.

Overall:

- The current step strongly supports that in high-ΔR² Populations there are many genes with strong spatial gradients beyond global Complexity and Purity.  
- The central open question (and the crux of your hypothesis) is whether the *orientation* of these gradients is reproducible across samples. Your next step is exactly the right move: per-sample refitting plus sign/angle concordance and per-sample significance.  
- If, for at least some Populations, you find a nontrivial subset of genes with:
  - high cross-sample sign concordance,  
  - strong and relatively stable gradient_norm, and  
  - frequent nominal per-sample significance,  
  that will be convincing evidence of intra-population, spatially organized, and reproducible maturation axes that are not reducible to global Complexity/Purity.

## Next Steps
Step 1: For each high-ΔR² Population identified from the Complexity~(Purity, Sample_ID, x, y) models, use the existing joint spatial tests and gradient norms to define robust spatial-gradient genes, then, within each such Population and gene, refit the linear model separately within each Sample_ID (cells restricted to that Population and sample; predictors: centered x, y, Complexity, Purity without Sample_ID fixed effects) to obtain per-sample β_x, β_y, gradient_norm, and a 2-df joint spatial F-test; finally, summarize and rank genes per Population by cross-sample sign concordance of β_x and β_y, median and IQR of gradient_norm, and fraction of samples with nominal spatial significance (p_xy < 0.05), and print text-only tables alongside each Population’s ΔR² and global spatial fraction to highlight the most reproducible, maturation-like spatial gradients.

## This code implements the final analysis step by refitting spatial linear models per gene within each sample for high-ΔR² Populations, then summarizing cross-sample consistency of robust spatial-gradient genes via sign concordance, gradient strength, and per-sample spatial significance, and finally ranking and printing text-only tables per Population.

In [ ]:
import numpy as np
import pandas as pd
from scipy import linalg, stats

# Step: Per-sample refitting and cross-sample consistency of robust spatial-gradient genes

# Required inputs from previous steps
complexity_df = adata.uns.get('complexity_spatial_models', None)
joint_by_pop = adata.uns.get('spatial_joint_stats_by_pop', None)
robust_by_pop = adata.uns.get('robust_spatial_gradient_genes_by_pop', None)

if complexity_df is None or joint_by_pop is None or robust_by_pop is None:
    raise ValueError("Missing required entries in adata.uns: need 'complexity_spatial_models', 'spatial_joint_stats_by_pop', and 'robust_spatial_gradient_genes_by_pop'.")

# High-ΔR² Populations (same criterion as before)
high_delta_threshold = 0.10
high_delta_pops = complexity_df.index[complexity_df['delta_r2'] >= high_delta_threshold].tolist()
if not high_delta_pops and not complexity_df.empty:
    # Fallback to top quartile if no population passes absolute cutoff
    q75 = complexity_df['delta_r2'].quantile(0.75)
    high_delta_pops = complexity_df.index[complexity_df['delta_r2'] >= q75].tolist()

obs = adata.obs.copy()
spatial = adata.obsm['spatial']
obs['x'] = spatial[:, 0]
obs['y'] = spatial[:, 1]

if not pd.api.types.is_categorical_dtype(obs['Populations']):
    obs['Populations'] = obs['Populations'].astype('category')
if not pd.api.types.is_categorical_dtype(obs['Sample_ID']):
    obs['Sample_ID'] = obs['Sample_ID'].astype('category')

# Use log1p-normalized expression as before
if 'log1p' in adata.layers:
    X_full = adata.layers['log1p']
else:
    X_full = adata.X
if not isinstance(X_full, np.ndarray):
    X_full = X_full.toarray()

var_names = np.array(adata.var_names)

per_sample_gene_stats = {}  # nested dict: {pop: DataFrame}

min_cells_per_sample = 30  # require at least this many cells for per-sample fits

for pop in high_delta_pops:
    robust_df = robust_by_pop.get(pop, None)
    if robust_df is None or robust_df.empty:
        # record explicit empty result so downstream can distinguish this case
        per_sample_gene_stats[pop] = pd.DataFrame(columns=[
            'n_samples_used','sign_concordance_x','sign_concordance_y',
            'median_grad_norm','IQR_grad_norm','frac_samples_pxy_lt_0.05',
            'composite_score'])
        print(f"\nPopulation {pop}: no robust spatial-gradient genes; skipping per-sample refits.")
        continue

    print(f"\nProcessing Population {pop} ...")

    # Cells belonging to this Population
    pop_mask = (obs['Populations'] == pop).values
    obs_pop = obs.loc[pop_mask, :].copy()
    X_pop = X_full[pop_mask, :]

    # Identify samples that have enough cells for this Population
    sample_counts = obs_pop['Sample_ID'].value_counts()
    usable_samples = sample_counts.index[sample_counts >= min_cells_per_sample].tolist()
    if len(usable_samples) < 2:
        print(f"  Skipping {pop}: fewer than 2 samples with >= {min_cells_per_sample} cells.")
        per_sample_gene_stats[pop] = pd.DataFrame(columns=[
            'n_samples_used','sign_concordance_x','sign_concordance_y',
            'median_grad_norm','IQR_grad_norm','frac_samples_pxy_lt_0.05',
            'composite_score'])
        continue

    print(f"  Usable samples for {pop}: {usable_samples}")

    # Genes to analyze: robust spatial-gradient genes in this Population
    genes_pop = robust_df.index.to_numpy()
    gene_idx = {g: i for i, g in enumerate(var_names)}
    gene_indices = np.array([gene_idx[g] for g in genes_pop if g in gene_idx], dtype=int)
    mapped_genes = var_names[gene_indices]

    if mapped_genes.size == 0:
        print(f"  No robust genes from {pop} found in adata.var_names; skipping.")
        per_sample_gene_stats[pop] = pd.DataFrame(columns=[
            'n_samples_used','sign_concordance_x','sign_concordance_y',
            'median_grad_norm','IQR_grad_norm','frac_samples_pxy_lt_0.05',
            'composite_score'])
        continue

    n_robust_total = len(genes_pop)
    n_mapped = mapped_genes.size
    n_dropped = n_robust_total - n_mapped
    if n_dropped > 0:
        print(f"  Warning: {n_dropped} of {n_robust_total} robust genes not found in adata.var_names and will be skipped.")

    # Containers to accumulate per-sample estimates for each gene
    beta_x_all = {g: [] for g in mapped_genes}
    beta_y_all = {g: [] for g in mapped_genes}
    grad_norm_all = {g: [] for g in mapped_genes}
    pval_xy_all = {g: [] for g in mapped_genes}
    samples_used_per_gene = {g: set() for g in mapped_genes}

    # Loop over samples and refit gene-wise models within each sample
    for sid in usable_samples:
        s_mask = (obs_pop['Sample_ID'] == sid).values
        n_s = int(s_mask.sum())
        if n_s < min_cells_per_sample:
            continue

        obs_ps = obs_pop.loc[s_mask, :].copy()
        X_ps = X_pop[s_mask, :][:, gene_indices]

        # Center numeric predictors within this Population+Sample
        for col in ['x', 'y', 'Complexity', 'Purity']:
            obs_ps[col] = obs_ps[col].astype(float)
            obs_ps[col] = obs_ps[col] - obs_ps[col].mean()

        # Design: intercept + x + y + Complexity + Purity
        design_df = pd.DataFrame({
            'intercept': np.ones(n_s, dtype=float),
            'x': obs_ps['x'].values,
            'y': obs_ps['y'].values,
            'Complexity': obs_ps['Complexity'].values,
            'Purity': obs_ps['Purity'].values,
        })
        X_design = design_df.values.astype(float)
        n_params = X_design.shape[1]
        dof = n_s - n_params
        if dof <= 0:
            print(f"  Sample {sid} in {pop}: insufficient dof (n={n_s}, p={n_params}); skipping.")
            continue

        # Precompute OLS quantities
        XtX = X_design.T @ X_design
        try:
            XtX_inv = linalg.inv(XtX)
        except linalg.LinAlgError:
            XtX_inv = linalg.pinv(XtX)
        Xt = X_design.T
        hat_part = XtX_inv @ Xt  # (p x n_s)

        # Fit all selected genes jointly in this sample
        beta_hat = hat_part @ X_ps  # (p x g_sel)
        Y_hat = X_design @ beta_hat
        resid = X_ps - Y_hat
        sigma2 = (resid ** 2).sum(axis=0) / dof  # (g_sel,)

        # Contrast for [x, y]
        param_names = list(design_df.columns)
        ix = param_names.index('x')
        iy = param_names.index('y')
        p = len(param_names)
        L = np.zeros((2, p))
        L[0, ix] = 1.0
        L[1, iy] = 1.0
        base_cov_xy = L @ XtX_inv @ L.T
        try:
            base_cov_xy_inv = np.linalg.inv(base_cov_xy)
        except np.linalg.LinAlgError:
            base_cov_xy_inv = np.linalg.pinv(base_cov_xy)

        # Extract x,y betas and compute joint F-test per gene
        beta_x = beta_hat[ix, :]
        beta_y = beta_hat[iy, :]
        grad_norm = np.sqrt(beta_x ** 2 + beta_y ** 2)

        F_vals = np.full(beta_x.shape[0], np.nan, dtype=float)
        p_vals = np.full(beta_x.shape[0], np.nan, dtype=float)

        for j in range(beta_x.shape[0]):
            if not np.isfinite(sigma2[j]) or sigma2[j] <= 0:
                continue
            b = np.array([beta_x[j], beta_y[j]])
            quad = float(b.T @ base_cov_xy_inv @ b)
            F = quad / (2.0 * sigma2[j])
            F_vals[j] = F
            p_vals[j] = 1.0 - stats.f.cdf(F, 2, dof)

        # Store per-sample stats per gene
        for g_idx, g in enumerate(mapped_genes):
            bx = beta_x[g_idx]
            by = beta_y[g_idx]
            gn = grad_norm[g_idx]
            pv = p_vals[g_idx]
            if not np.isfinite(bx) or not np.isfinite(by) or not np.isfinite(gn) or not np.isfinite(pv):
                continue
            beta_x_all[g].append(bx)
            beta_y_all[g].append(by)
            grad_norm_all[g].append(gn)
            pval_xy_all[g].append(pv)
            samples_used_per_gene[g].add(sid)

    # Summarize cross-sample statistics for this Population's robust genes
    rows = []
    for g in mapped_genes:
        bx_list = np.array(beta_x_all[g], dtype=float)
        by_list = np.array(beta_y_all[g], dtype=float)
        gn_list = np.array(grad_norm_all[g], dtype=float)
        pv_list = np.array(pval_xy_all[g], dtype=float)
        n_samp = bx_list.size
        if n_samp == 0:
            continue

        # Sign concordance (relative to majority sign among samples)
        def sign_concordance(vals):
            s = np.sign(vals)
            s = s[np.abs(s) > 0]  # ignore exact zeros
            if s.size == 0:
                return 0.0
            maj = 1.0 if (s.sum() >= 0) else -1.0
            return float(np.mean(s == maj))

        sc_x = sign_concordance(bx_list)
        sc_y = sign_concordance(by_list)

        # Gradient magnitude across samples
        med_gn = float(np.median(gn_list))
        iqr_gn = float(np.percentile(gn_list, 75) - np.percentile(gn_list, 25)) if gn_list.size > 1 else 0.0

        # Fraction of samples with nominally significant spatial test
        frac_sig = float(np.mean(pv_list < 0.05)) if pv_list.size > 0 else 0.0

        # Composite score combining concordance, strength, and per-sample significance
        concordance_mean = 0.5 * (sc_x + sc_y)
        composite_score = concordance_mean * med_gn * frac_sig

        rows.append({
            'gene': g,
            'n_samples_used': n_samp,
            'sign_concordance_x': sc_x,
            'sign_concordance_y': sc_y,
            'median_grad_norm': med_gn,
            'IQR_grad_norm': iqr_gn,
            'frac_samples_pxy_lt_0.05': frac_sig,
            'composite_score': composite_score,
        })

    if not rows:
        print(f"  No valid per-sample fits for robust genes in {pop}.")
        per_sample_gene_stats[pop] = pd.DataFrame(columns=[
            'n_samples_used','sign_concordance_x','sign_concordance_y',
            'median_grad_norm','IQR_grad_norm','frac_samples_pxy_lt_0.05',
            'composite_score'])
        continue

    stats_df = pd.DataFrame(rows).set_index('gene')
    # Rank genes by composite score (descending)
    stats_df.sort_values('composite_score', ascending=False, inplace=True)
    per_sample_gene_stats[pop] = stats_df

    # Print a concise textual summary for this Population
    delta_r2 = float(complexity_df.loc[pop, 'delta_r2']) if pop in complexity_df.index else np.nan
    frac_sig_global = np.nan
    if pop in joint_by_pop:
        jdf = joint_by_pop[pop]
        valid = np.isfinite(jdf['fdr_xy'])
        n_tot = int(valid.sum())
        n_sig = int(((jdf['fdr_xy'] < 0.05) & valid).sum())
        frac_sig_global = (n_sig / n_tot) if n_tot > 0 else np.nan

    print(f"\nPopulation {pop}: ΔR² (Complexity | x,y) = {delta_r2:.3f}, global frac(spatial FDR<0.05) = {frac_sig_global:.3f}")
    print("Top 10 robust, cross-sample-consistent spatial-gradient genes:")
    print(stats_df.head(10)[['n_samples_used', 'sign_concordance_x', 'sign_concordance_y',
                             'median_grad_norm', 'IQR_grad_norm', 'frac_samples_pxy_lt_0.05',
                             'composite_score']])

# Store the per-sample summary statistics in adata.uns for later inspection or reporting
adata.uns['per_sample_spatial_gradient_stats_by_pop'] = per_sample_gene_stats



Processing Population PA ...
  Usable samples for PA: ['R78_4C15', 'R78_4C12', 'R77_4C4']

Population PA: ΔR² (Complexity | x,y) = 0.148, global frac(spatial FDR<0.05) = 0.857
Top 10 robust, cross-sample-consistent spatial-gradient genes:
          n_samples_used  sign_concordance_x  sign_concordance_y  \
gene                                                               
SOX9                   3                 1.0            1.000000   
SLC1A3                 3                 1.0            1.000000   
DES                    3                 1.0            1.000000   
IRX2                   3                 1.0            1.000000   
RABGAP1L               3                 1.0            1.000000   
IRX1                   3                 1.0            1.000000   
APOE                   3                 1.0            1.000000   
CNN1                   3                 1.0            1.000000   
TNNT1                  3                 1.0            0.666667   
PLN         


Population PB: ΔR² (Complexity | x,y) = 0.102, global frac(spatial FDR<0.05) = 0.908
Top 10 robust, cross-sample-consistent spatial-gradient genes:
        n_samples_used  sign_concordance_x  sign_concordance_y  \
gene                                                             
MYH7                 3                 1.0            1.000000   
CXCL12               3                 1.0            1.000000   
SFRP1                3                 1.0            1.000000   
DHRS3                3                 1.0            0.666667   
NR2F1                3                 1.0            0.666667   
MAF                  3                 1.0            0.666667   
CASQ2                3                 1.0            0.666667   
GJA1                 3                 1.0            1.000000   
PLN                  3                 1.0            0.666667   
PAM                  3                 1.0            1.000000   

        median_grad_norm  IQR_grad_norm  frac_samples_pxy_


Population PE: ΔR² (Complexity | x,y) = 0.153, global frac(spatial FDR<0.05) = 0.798
Top 10 robust, cross-sample-consistent spatial-gradient genes:
        n_samples_used  sign_concordance_x  sign_concordance_y  \
gene                                                             
IRX1                 3            1.000000                 1.0   
IRX2                 3            1.000000                 1.0   
PRSS35               3            1.000000                 1.0   
GJA1                 3            0.666667                 1.0   
CASQ2                3            1.000000                 1.0   
CGNL1                3            1.000000                 1.0   
PLN                  3            0.666667                 1.0   
TBX3                 3            1.000000                 1.0   
CXCL12               3            1.000000                 1.0   
GJA5                 3            1.000000                 1.0   

        median_grad_norm  IQR_grad_norm  frac_samples_pxy_


Population PJ: ΔR² (Complexity | x,y) = 0.127, global frac(spatial FDR<0.05) = 0.681
Top 10 robust, cross-sample-consistent spatial-gradient genes:
       n_samples_used  sign_concordance_x  sign_concordance_y  \
gene                                                            
PLK2                3                 1.0            1.000000   
PLN                 3                 1.0            1.000000   
CKMT2               3                 1.0            1.000000   
FBLN5               3                 1.0            1.000000   
SOX9                3                 1.0            0.666667   
ABCC9               3                 1.0            1.000000   
CASQ2               3                 1.0            1.000000   
DHRS3               3                 1.0            1.000000   
CNN1                3                 1.0            1.000000   
RBP1                3                 1.0            1.000000   

       median_grad_norm  IQR_grad_norm  frac_samples_pxy_lt_0.05  \
ge


Population PP: ΔR² (Complexity | x,y) = 0.126, global frac(spatial FDR<0.05) = 0.672
Top 10 robust, cross-sample-consistent spatial-gradient genes:
       n_samples_used  sign_concordance_x  sign_concordance_y  \
gene                                                            
PLK2                3            1.000000                 1.0   
SOX9                3            0.666667                 1.0   
DES                 3            1.000000                 1.0   
PLN                 3            1.000000                 1.0   
DCN                 3            1.000000                 1.0   
ABCC9               3            1.000000                 1.0   
IRX3                3            1.000000                 1.0   
CAV1                3            1.000000                 1.0   
IRX4                3            1.000000                 1.0   
TBX18               3            0.666667                 1.0   

       median_grad_norm  IQR_grad_norm  frac_samples_pxy_lt_0.05  \
ge

### Agent Interpretation

These per-sample fits are strongly supportive of the hypothesis and give you a very rich set of candidate maturation axes to exploit in the next steps.

Key points and how I’d build on them:

1. **Hypothesis support: robust, cross-sample spatial gradients within high-ΔR² populations**

   - Many high-ΔR² pops (PA, PB, PC, PD, PE, PH, PI, PJ, PK, PL, PO, PP, PQ, PV, PW, PAA) show:
     - ΔR² ≳ 0.10 and high global spatial fractions (often >0.7–0.8).
     - For robust spatial-gradient genes, **n_samples_used = 3** (or 2) with:
       - **sign_concordance_x and sign_concordance_y ≥ 0.66–1.0**, i.e., consistent gradient direction across samples.
       - **frac_samples_pxy_lt_0.05 = 1.0** for almost all top genes, meaning each gene’s spatial component remains significant in (nearly) every sample after adjusting for Complexity and Purity.
       - Non-negligible **median_grad_norm** with modest IQRs.
   - This is exactly the pattern the hypothesis calls for: robust, reproducible spatial gradients within populations that exist **after controlling for Complexity, Purity, and sample effects**, and with consistent orientation.

   - Repeatedly appearing genes (e.g. PLN, MYH6, MYH7, DES, GJA1, GJA5, IRX1/2/3/4, TBX3, POSTN, CXCL12, IGFBP4/5, LBH, FBLN2/5, CASQ2, PRSS35, DHRS3, SFRP1, etc.) across multiple pops are especially compelling as components of **shared or aligned maturation axes across related populations**.

2. **Promising populations and gene sets**

   Focus populations where ΔR² is high and the top genes show strong, concordant gradients:

   - **Populations with very strong ΔR² and spatial fraction**:
     - PA (ΔR²=0.148, frac≈0.86): SOX9, SLC1A3, DES, IRX1/2, APOE, CNN1, TNNT1, PLN.
     - PC (ΔR²=0.144, frac≈0.85): IGFBP4, PLN, IRX3/4, DHRS3, SLC1A3, SOX9, DES.
     - PD (ΔR²=0.154, frac≈0.85): PLN, IGFBP4/5, DHRS3, CKMT2, RYR2, POSTN, CXCL12.
     - PE (ΔR²=0.153, frac≈0.80): IRX1/2, PRSS35, GJA1, CASQ2, CGNL1, PLN, TBX3, CXCL12, GJA5.
     - PQ (ΔR²=0.151, frac≈0.72): MYH6, GJA1/5, IRX1/3, TBX3, IGFBP5, LBH, PLN, MSX2.
     - PW (ΔR²=0.159, frac≈0.60): MYH7, GJA5, DHRS3, TBX3, PAM, OSR1, CXCL12, VCAN, VSNL1, TRPM3.

   - **Pops with especially large gradient norms for classic cardiac genes**:
     - **PQ, PW, PV, PAA** show MYH6/MYH7/GJA5/TBX3/HAND2 with quite large median_grad_norm values (10⁻³ range), suggesting pronounced spatial change along a potential maturation or regional axis.
     - PV and PW, despite having only 2 samples, have very strong gradients for developmental/region markers (e.g., HAND2, TBX3, NR2F1, MSX2, BMP2, DKK3, COL2A1), ideal for hypothesizing anatomical axes.

   - Repeated patterns:
     - **PLN and CASQ2** show up in many ventricular-like populations (contractility / Ca²⁺ handling).
     - **IRX1/2/3/4 and TBX3** recur in multiple pops with consistent gradients, suggesting a robust axis of conduction system / boundary-region maturation.
     - **CXCL12, SFRP1, DHRS3, IGFBP4/5, FBLN2/5, POSTN** hint at coordinated spatial variation in ECM, signaling, and possibly border tissues.

3. **Next steps: defining and validating intra-population maturation axes**

   a. **Treat the per-population top genes as axis “signatures”**

   - For each high-ΔR² population:
     - Take the top N genes by composite_score (e.g. 20–50).
     - Optionally weight them by composite_score or median_grad_norm.
     - Construct a **per-cell “spatial gradient score”** as a weighted sum or first PC of these genes’ expressions (within that population).
   - Because β_x, β_y are sign-consistent across samples, this score should increase in the same spatial direction across sections, i.e., a robust intra-population axis.

   b. **Quantify directionality consistency explicitly**

   - Combine the per-sample (β_x, β_y) for each gene into a population-level unit vector:
     - For each pop, for each gene g, compute unit vectors `u_s = (β_x, β_y)/||β||` per sample, then average across samples and re-normalize to get ū_g.
     - Then average ū_g across top genes to get a **population-level axis direction** ū_pop.
   - This gives a single 2D spatial axis per population (like a “maturation vector”). Verify that per-sample axes align well with ū_pop (e.g., dot product > 0.7 for most genes and samples).

   c. **Project cells onto the axis**

   - For each cell in a pop, compute projected coordinate `t = x * ū_pop[0] + y * ū_pop[1]` (after centering within sample).
   - Regress gene expression on `t`, Complexity, Purity within each sample:
     - This tests whether the gene gradients are captured by a **single 1D axis** rather than arbitrary 2D patterns.
     - It will also let you compare **effect sizes along t vs. Complexity**.

   d. **Test reproducibility across samples**

   - For each population:
     - For each gene, correlate its expression with t in each sample, and summarize:
       - cross-sample sign concordance of correlation with t,
       - median |correlation|,
       - fraction of samples with nominal significance (FDR for correlation vs. t).
     - This is analogous to what you just did with (x, y), but using the **shared axis**.

   - If a large set of genes preserve direction and significance along t across samples, that’s strong evidence for reproducible intra-population axes.

4. **Relating axes across populations (shared vs. population-specific maturation)**

   - Several genes appear across multiple pops (PLN, MYH6/7, DES, IGFBP4/5, IRX3/4, GJA1/5, TBX3, LBH, etc.).
   - For each shared gene:
     - Compare per-population **median_grad_norm** and sign patterns; check if the gene increases along the same global anatomical direction (after aligning population-level axes to a common reference, e.g. canonical x,y orientation).
   - Use **gene overlap of top lists** to cluster populations:
     - Compute Jaccard or overlap of top 30 genes per pop (by composite_score).
     - This can reveal groups of pops that share a maturation axis vs. those with distinct axes (e.g. atrial-like vs ventricular-like vs conduction-like clusters).

5. **Integrating with Complexity / Purity without duplicating past analyses**

   - Your model already adjusts for Complexity and Purity at per-sample level, but you can now test:
     - Within each pop, whether **projected coordinate t explains residual expression variance** after accounting for Complexity and Purity.
     - Compare the variance explained by t vs. by Complexity; you’re asking whether this spatial axis is an **independent maturation dimension**.
   - Avoiding duplication with previous work:
     - Past Analysis 1 contrasted high/low Complexity tertiles; here you’re focusing on **spatially aligned, per-population axes** independent of Complexity.
     - Past Analysis 2 focused on neighborhood-composition changes; your current axis is **within-population transcriptional and spatial structure**, not neighbor composition.

6. **Biological interpretability (within the constraints)**

   - Without assigning true identities, you can still:
     - Examine per-population average expression of these top gradient genes to hypothesize which pops are progenitor-like vs. more mature contractile vs. conduction vs. fibroblast/ECM-like.
     - For each population, inspect how expression of canonical structural/ECM/signaling genes changes along t; this can be phrased as “proximal vs. distal regions in the developing heart”.

7. **Potential robustness checks**

   - **Rescaling of x,y**: your gradients depend on coordinate scale; ensure all samples are in a common coordinate frame or at least that per-sample centering (already done) is adequate. If sections differ in scale, consider standardizing x and y to unit variance per sample before fitting.
   - **Gene selection robustness**:
     - Repeat the per-sample modeling restricting to genes with sufficient detection rates within each pop to avoid sparse-expression artifacts.
   - **Nonlinearity**:
     - For a few key genes/pops, plot smoothed expression vs. t within each sample to confirm roughly monotonic gradients and rule out strongly non-linear or patchy patterns masquerading as linear gradients.

8. **Concrete immediate next steps**

   - For 3–5 representative populations (e.g. PA, PE, PK, PQ, PAA):
     1. Derive ū_pop and t as above.
     2. Plot:
        - cells colored by t (and by Complexity) to visually verify a smooth intra-population axis.
        - expression of top 5–10 gradient genes vs. t, per sample (lines or smoothers).
     3. Compute and tabulate:
        - per-gene correlation with t, sign concordance across samples.
        - variance explained by t vs. Complexity (R² comparison) at gene and population levels.

   - Then:
     - Summarize per-population “axis gene sets” (top ~30 genes) and a smaller **core set of cross-population recurrent axis genes**.
     - Use these for downstream biological interpretation and for comparing axes across developmental stages or regions (if such metadata exist).

Overall, the current results very much validate the existence of robust, population-specific and partly shared intra-population spatial axes that persist after adjusting for Complexity and Purity. The next logical step is to collapse the 2D (x,y) information into well-defined 1D axes per population and quantify how gene expression varies along those axes across samples, thereby turning these per-sample β estimates into interpretable maturation gradients.